# Figure 2 historical evaluation workflow

This public copy retains the historical Figure 2 extraction and evaluation cells, together with their saved outputs. Absolute storage paths and API credentials were replaced, and unrelated downstream analyses were omitted. The evaluation logic and archived prediction files were not changed.

All required files are available in `./data/`. Extraction cells require users to supply their own API credentials.


# Import

In [ ]:
!pip install ipython-autotime
!pip install kaplanmeier
!pip install openai
!pip install bert_score
# !pip install scispacy
# !pip install spacy
# !pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 11.2 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=cf14c8e23ec279972ce8d2291dc4e0200fb63cc43249a571a0f7163c939a08bc
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
import codecs
import xml.etree.ElementTree as ET
import sys
import requests
from scipy.stats import chi2_contingency
from scipy.stats import chi2_contingency
import numpy as np
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import ast
import difflib
import zipfile
import ast
from collections import Counter
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from ast import literal_eval
import re
#import kaleido
from tqdm import tqdm
import scipy.stats as stats
from lifelines import KaplanMeierFitter
import bert_score
from multiprocessing import Pool, cpu_count
from concurrent.futures import ThreadPoolExecutor
import threading
from queue import Queue
from bert_score import score

import time

%load_ext autotime

time: 330 µs (started: 2025-11-04 08:02:19 +00:00)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 4.86 s (started: 2025-11-04 08:02:19 +00:00)


In [ ]:
from openai import OpenAI

time: 494 µs (started: 2025-01-15 08:12:50 +00:00)


In [ ]:
MODEL="gpt-4o"
MODEL="deepseek-chat"
#openai
#client = OpenAI(api_key='YOUR_API_KEY')

#chatnio
#client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.chatnio.net/v1")

#deepseek
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.deepseek.com")

time: 53.1 ms (started: 2024-07-24 11:58:06 +00:00)


# NC validation dataset

In [ ]:
nc_input_df = pd.read_csv('./data/nc_pmids_result.csv').drop(columns=['Unnamed: 0'])
#('./data/nc_pmids_result.csv').drop(columns=['Unnamed: 0'])

print(nc_input_df.shape,nc_input_df.columns)
nc_input_df['input'] = "Title: " + nc_input_df['article_title'] + '. Abstract: ' + nc_input_df['full_abs']
nc_input_df['pmid'] = nc_input_df['pmid'].astype(int).astype(str)
#nc_input_df  = nc_input_df.dropna(subset=['input'])
print(nc_input_df.shape,nc_input_df.columns)

input_df = nc_input_df[~nc_input_df['pmid'].isin(['8688759','17379620'])]
print(input_df['pmid'].nunique())

(78, 17) Index(['pmid', 'pub_year', 'pub_month', 'epub_year', 'epub_month', 'pub_types',
       'full_abs', 'article_title', 'background', 'method', 'result',
       'conclusion', 'label_result', 'label_conclusion', '# of comments',
       'all_mesh_terms', 'major_mesh_terms'],
      dtype='object')
(78, 18) Index(['pmid', 'pub_year', 'pub_month', 'epub_year', 'epub_month', 'pub_types',
       'full_abs', 'article_title', 'background', 'method', 'result',
       'conclusion', 'label_result', 'label_conclusion', '# of comments',
       'all_mesh_terms', 'major_mesh_terms', 'input'],
      dtype='object')
76
time: 850 ms (started: 2024-09-01 23:42:34 +00:00)


In [ ]:
input_df.sample(2)

,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input
36,25246449,2014.0,Sep,2014.0,9.0,"['Journal Article', 'Research Support, N.I.H.,...",The healthiest dietary pattern for myocardial ...,Low carbohydrate diet from plant or animal sou...,The healthiest dietary pattern for myocardial ...,NaN,We included 2258 women from the Nurses' Health...,Greater adherence to an LCD high in animal sou...,We included 2258 women from the Nurses' Health...,Greater adherence to an LCD high in animal sou...,NaN,"['Adult', 'Animals', 'Cardiovascular Diseases/...","['Cause of Death*', 'Diet, Carbohydrate-Restri...",Title: Low carbohydrate diet from plant or ani...
4,19211817,2009.0,May,2009.0,2.0,"['Journal Article', 'Multicenter Study', 'Rese...",Saturated fatty acid (SFA) intake increases pl...,Major types of dietary fat and risk of coronar...,Saturated fatty acid (SFA) intake increases pl...,This was a follow-up study in which data from ...,"During 4-10 y of follow-up, 5249 coronary even...",The associations suggest that replacing SFAs w...,"During 4-10 y of follow-up, 5249 coronary even...",The associations suggest that replacing SFAs w...,2.0,"['Adult', 'Aged', 'Cholesterol, LDL/blood/drug...","['Coronary Disease/epidemiology*', 'Dietary Fa...",Title: Major types of dietary fat and risk of ...


time: 66.7 ms (started: 2024-07-31 01:34:29 +00:00)


In [ ]:
nc_vali_df = pd.read_excel('./data/nc_figure1.xlsx').rename(columns={'Category':'category','Exposure':'exposure','Label':'label','Phenotype':'outcome_phenotype','Association':'association','Effect Size':'effect_size','PubMed_ID':'pmid'})
print(nc_vali_df.shape,nc_vali_df.columns,nc_vali_df['pmid'].nunique())
print(nc_vali_df['category'].unique())
print(nc_vali_df['outcome_phenotype'].unique())

nc_vali_df = nc_vali_df.dropna(subset=['pmid'])

nc_vali_df['pmid'] = nc_vali_df['pmid'].astype(int).astype(str)

nc_vali_df = nc_vali_df[nc_vali_df['pmid'].isin(input_df['pmid'])]
print(nc_vali_df.shape,nc_vali_df.columns,nc_vali_df['pmid'].nunique())

nc_vali_df['association'] = nc_vali_df['association'].replace({'neg':'negative','pos':'positive'})
nc_vali_df.sample(3)

(291, 7) Index(['category', 'exposure', 'label', 'outcome_phenotype', 'association',
       'effect_size', 'pmid'],
      dtype='object') 78
['Food/Diet' 'Lipid' 'Mineral' 'Vitamin' 'Carb' 'Alcohol' 'Fiber' 'Other'
 'Protein']
['CHD' 'MI' 'Hemorrhagic stroke' 'Ischemic stroke' 'Total stroke'
 'Ischemic heart disease' 'Nonfatal MI' 'Sudden cardiac death' 'CVD'
 'Peripheral artery disease' 'Coronary artery disease' 'Thrombotic stroke'
 'Hypertension' 'Atherothrombotic stroke' 'Cardioembolic stroke']
(249, 7) Index(['category', 'exposure', 'label', 'outcome_phenotype', 'association',
       'effect_size', 'pmid'],
      dtype='object') 76


,category,exposure,label,outcome_phenotype,association,effect_size,pmid
137,Food/Diet,egg,Egg,Total stroke,no,NaN,23295181
221,Lipid,omega-3 biomarker α-linolenic acid,ALA biomarker,CHD,negative,0.91,27357102
167,Other,cystatin C,Cystatin C,Peripheral artery disease,positive,1.54,24980133


time: 2.64 s (started: 2024-07-31 01:34:29 +00:00)


In [ ]:
test_input_1 = """ Title: Fruit and Vegetable Consumption and the Incidence of Hypertension in Three Prospective Cohort Studies. Abstract: Increased fruit and vegetable intake lowers blood pressure in short-term interventional studies. However, data on the association of long-term intake of fruits and vegetables with hypertension risk are scarce. We prospectively examined the independent association of whole fruit (excluding juices) and vegetable intake, as well as the change in consumption of whole fruits and vegetables, with incident hypertension in 3 large longitudinal cohort studies: Nurses' Health Study (n=62 175), Nurses' Health Study II (n=88 475), and Health Professionals Follow-up Study (n=36 803). We calculated hazard ratios and 95% confidence intervals for fruit and vegetable consumption while controlling for hypertension risk factors. Compared with participants whose consumption was ≤4 servings/week, the pooled hazard ratios among those whose intake was ≥4 servings/day were 0.92(0.87-0.97) for total whole fruit intake and 0.95(0.86-1.04) for total vegetable intake. Similarly, compared with participants who did not increase their fruit or vegetable consumption, the pooled hazard ratios for those whose intake increased by ≥7 servings/week were 0.94(0.90-0.97) for total whole fruit intake and 0.98(0.94-1.01) for total vegetable. Analyses of individual fruits and vegetables yielded different results. Consumption levels of ≥4 servings/week (as opposed to <1 serving/month) of broccoli, carrots, tofu or soybeans, raisins, and apples was associated with lower hypertension risk. In conclusion, our results suggest that greater long-term intake and increased consumption of whole fruits may reduce the risk of developing hypertension.
"""
test_input_2 = """ Title: Dietary intake of marine n-3 fatty acids, fish intake, and the risk of coronary disease among men. Abstract: Background: It has been hypothesized that a diet containing n-3 fatty acids from fish reduces the risk of coronary heart disease, but few large epidemiologic studies have examined this question.

Methods: In 1986, 44,895 male health professionals, 40 to 75 years of age, who were free of known cardiovascular disease completed detailed and validated dietary questionnaires as part of the Health Professionals Follow-up Study. During six years of follow-up, we documented 1543 coronary events in this group: 264 deaths from coronary disease, 547 nonfatal myocardial infarctions, and 732 coronary-artery bypass or angioplasty procedures.

Results: After controlling for age and several coronary risk factors, we observed no significant associations between dietary intake of n-3 fatty acids or fish intake and the risk of coronary disease. For men in the top fifth of the group in terms of intake of n-3 fatty acids (median, 0.58 g per day), the multivariate relative risk of coronary heart disease was 1.12 (95 percent confidence interval, 0.96 to 1.31), as compared with the men in the bottom fifth (median, 0.07 g per day). For men who consumed six or more servings of fish per week, as compared with those who consumed one serving per month or less, the multivariate relative risk of coronary disease was 1.14 (95 percent confidence interval, 0.86 to 1.51). The risk of death due to coronary disease among men who ate any amount of fish, as compared with those who ate no fish, was 0.74 (95 percent confidence interval, 0.44 to 1.23), but the risk did not decrease as fish consumption increased.

Conclusions: Although the possibility of residual confounding by unmeasured factors cannot be entirely excluded, these data suggest that increasing fish intake from one to two servings per week to five to six servings per week does not substantially reduce the risk of coronary heart disease among men who are initially free of cardiovascular disease.
"""

test_input_3 = """ Title: Relation of consumption of vitamin E, vitamin C, and carotenoids to risk for stroke among men in the United States

Abstract: Background: Antioxidants increase the resistance of low-density lipoprotein to oxidation and may thereby reduce risk for atherosclerosis.

Objective: To determine whether intake of vitamin E, vitamin C, or carotenoids predict risk for total or ischemic stroke.

Design: Prospective observational study.

Setting: The Health Professionals Follow-up Study.

Participants: 43,738 men 40 to 75 years of age who did not have cardiovascular disease or diabetes.

Measurements: Repeated and validated dietary assessments were done by using a self-administered 131-item food-frequency questionnaire, which included questions on dose and duration of vitamin supplement use. The follow-up period was 8 years.

Results: A total of 328 strokes occurred: 210 ischemic, 70 hemorrhagic, and 48 unclassified. After adjustment for age, smoking, hypertension, hypercholesterolemia, body mass index, physical activity, parental history of myocardial infarction, alcohol consumption, and total energy intake, the relative risk for ischemic stroke in the top quintile of vitamin E intake (median, 411 IU/d) compared with the bottom quintile (5.4 IU/d) was 1.18 (95% CI, 0.77 to 1.82). The relative risk for ischemic stroke in the top quintile of vitamin C intake (1167 mg/d) compared with the bottom quintile (95 mg/d) was 1.03 (CI, 0.66 to 1.59). Results for total stroke were similar. Associations of vitamin intake with hemorrhagic stroke were also nonsignificant, but the CIs were wide. Neither dose nor duration of vitamin E or vitamin C supplement use was related to risk for total or ischemic stroke. The relative risk for ischemic stroke was 1.16 (CI, 0.81 to 1.67) in men using 250 IU or more of vitamin E supplementation per day compared with men who used no vitamin E supplements and was 0.93 (CI, 0.60 to 1.45) in men using 700 mg or more of vitamin C supplementation per day compared with men who used no vitamin C supplements. A significant inverse relation between lutein intake and risk for ischemic stroke was seen but was not independent of other dietary factors.

Conclusions: Vitamin E and vitamin C supplements and specific carotenoids did not seem to substantially reduce risk for stroke in this cohort. Modest effects, however, cannot be excluded.
"""

time: 486 µs (started: 2024-07-31 01:34:31 +00:00)


In [ ]:
nc_vali_df.to_csv('./data/nc_vali_df.csv')

time: 1.07 s (started: 2024-07-25 01:58:34 +00:00)


## prompt #1

In [ ]:
prompt = """

From the following given title and abstract, extract the associations between dietary exposures and cardiovascular complications, such as coronary heart disease, stroke, and hypertension.
*More than one association might be studied in a paper. Extract all associations (one exposure to one outcome pairs)

fyi: CHD refers to non-fatal MI and fatal coronary heart disease; also, coronary artery disease (CAD) refers to non-fatal MI and fatal coronary artery disease. Cardiovascular disease (CVD) is defined as a composite of coronary artery disease and non-fatal or fatal stroke.

extract following information:

1. "exposure": the studied exposure
2. "label": standardized exposure concept
3. "category": the category of exposure based on 9 categories: 'Food/Diet' 'Lipid' 'Mineral' 'Vitamin' 'Carb' 'Alcohol' 'Fiber' 'Other' 'Protein'
4. "outcome_phenotype": the studied outcome, which is a cardiovascular related disease such as CHD, stroke, etc. only use abbreviation for CHD, MI, and CVD. return full name for others
5. "association": the association between exposure and outcome, which is either positive or negative or no association
6. "effect_size": the effect size of the association, only return for positive and negative associations. return "not found" for no association (output number, do not output percentage)

Return the extracted information in a list of flat JSON dictionaries of each result.

example input:
""
Title: "Dietary fat and risk of coronary heart disease in men: cohort follow up study in the United States"

Abstract: "Objective: To examine the association between fat intake and the incidence of coronary heart disease in men of middle age and older.
Design: Cohort questionnaire study of men followed up for six years from 1986.
Setting: The health professionals follow up study in the United States.
Subjects: 43 757 health professionals aged 40 to 75 years free of diagnosed cardiovascular disease or diabetes in 1986.
Main outcome measure: Incidence of acute myocardial infarction or coronary death.
Results: During follow up 734 coronary events were documented, including 505 non-fatal myocardial infarctions and 229 deaths. After age and several coronary risk factors were controlled for significant positive associations were observed between intake of saturated fat and risk of coronary disease. For men in the top versus the lowest fifth of saturated fat intake (median = 14.8% v 5.7% of energy) the multivariate relative risk for myocardial infarction was 1.22 (95% confidence interval 0.96 to 1.56) and for fatal coronary heart disease was 2.21 (1.38 to 3.54). After adjustment for intake of fibre the risks were 0.96 (0.73 to 1.27) and 1.72 (1.01 to 2.90), respectively. Positive associations between intake of cholesterol and risk of coronary heart disease were similarly attenuated after adjustment for fibre intake. Intake of linolenic acid was inversely associated with risk of myocardial infarction; this association became significant only after adjustment for non-dietary risk factors and was strengthened after adjustment for total fat intake (relative risk 0.41 for a 1% increase in energy, P for trend < 0.01)."
Conclusions: These data do not support the strong association between intake of saturated fat and risk of coronary heart disease suggested by international comparisons. They are compatible, however, with the hypotheses that saturated fat and cholesterol intakes affect the risk of coronary heart disease as predicted by their effects on blood cholesterol concentration. They also support a specific preventive effect of linolenic acid intake. "
""

example output:
[{
    "exposure": "linolenic acid",
    "label": "Linolenic acid",
    "category": "Lipid",
    "outcome_phenotype": "MI",
    "association": "negative",
    "effect_size": "0.41"
},
{
    "exposure": "saturated fat",
    "label": "SFA",
    "category": "Lipid",
    "outcome_phenotype": "CHD",
    "association": "positive",
    "effect_size": "2.21"
},
{
    "exposure": "saturated fat",
    "label": "SFA",
    "category": "Lipid",
    "outcome_phenotype": "MI",
    "association": "positive",
    "effect_size": "1.22"
},
{
    "exposure": "saturated fat adjusted for fiber",
    "label": "SFA adjusted for fiber",
    "category": "Lipid",
    "outcome_phenotype": "CHD",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "saturated fat adjusted for fiber",
    "label": "SFA adjusted for fiber",
    "category": "Lipid",
    "outcome_phenotype": "MI",
    "association": "no",
    "effect_size": "not found"
}
]

still return a list of dictionaries with empty values if no association is found or input is irrelevant
Only output in json format, do not include any explanations or additional text in your response.
*Dont output ```json ``` at the beginning nor end
"""


test_prompt = prompt

system_1 = 'You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract strucutured information from free texts of article titles and abstracts"'

time: 464 µs (started: 2024-07-25 01:58:36 +00:00)


In [ ]:
import openai
import pandas as pd
from tqdm import tqdm
from ast import literal_eval
import threading
from queue import Queue
import json

# Function to extract primary results
def extract_primary_result(text, prompt):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_1},
            {"role": "user", "content": prompt},
            {"role": "user", "content": text}
        ],
        temperature=0
    )
    return completion.choices[0].message['content']

# Function to perform the two-step extraction
def two_step_extraction(text):
    # Step 1: Extract exposures and outcomes
    completion_step1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
            {"role": "user", "content": prompt_step1},
            {"role": "user", "content": text}
        ]
    )
    entities_result = completion_step1.choices[0].message['content']

    entities_dict = json.loads(entities_result)
    entities_json = json.dumps(entities_dict, indent=2)

    # Prepare step 2 prompt
    prompt_step2 = prompt_step2_template.format(entities=entities_json)

    # Step 2: Extract associations using the entities
    completion_step2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
            {"role": "user", "content": prompt_step2},
            {"role": "user", "content": f"extracted entities: {entities_json}"},
            {"role": "user", "content": f"title and abstract: {text}"}
        ]
    )
    associations_result = completion_step2.choices[0].message['content']

    return associations_result

# Worker function to process each row
def worker(input_queue, output_queue, text_column, progress_bar):
    while True:
        index, row = input_queue.get()
        if index is None:
            break
        try:
            output = extract_primary_result(row[text_column])
            extracted_results = literal_eval(output)
            for res in extracted_results:
                res['pmid'] = row['pmid']
                output_queue.put(res)
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            print(f"Input text: {row[text_column][:500]}...")  # Print the first 500 characters of the text
            print(f"Output: {output}")  # Print the output if available
        finally:
            input_queue.task_done()
            progress_bar.update(1)

# Main function to compile extracted results
def compile_extracted_results(input_df, text_column):
    results = []
    input_queue = Queue()
    output_queue = Queue()

    # Progress bar for processing
    progress_bar = tqdm(total=input_df.shape[0], desc="Processing abstracts")

    # Start worker threads
    for _ in range(8):
        threading.Thread(target=worker, args=(input_queue, output_queue, text_column, progress_bar), daemon=True).start()

    # Enqueue rows for processing
    for index, row in input_df.iterrows():
        input_queue.put((index, row))

    # Wait for all tasks to be processed
    input_queue.join()

    # Collect results from output queue
    while not output_queue.empty():
        results.append(output_queue.get())

    # Close progress bar
    progress_bar.close()

    results_df = pd.json_normalize(results)
    return results_df

time: 2.94 ms (started: 2024-07-26 05:44:47 +00:00)


In [ ]:
input_df = nc_input_df[~nc_input_df['pmid'].isin(['8688759','17379620'])]
print(input_df['pmid'].nunique())

76
time: 2.38 ms (started: 2024-07-25 01:58:39 +00:00)


### deepseek

In [ ]:
MODEL="deepseek-chat"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.deepseek.com")

time: 53.1 ms (started: 2024-07-24 15:32:25 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[
    {
        "exposure": "n-3 fatty acids",
        "label": "n-3 fatty acids",
        "category": "Lipid",
        "outcome_phenotype": "CHD",
        "association": "no",
        "effect_size": "not found"
    },
    {
        "exposure": "fish intake",
        "label": "Fish intake",
        "category": "Food/Diet",
        "outcome_phenotype": "CHD",
        "association": "no",
        "effect_size": "not found"
    }
]
time: 10.1 s (started: 2024-07-24 15:32:25 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_3},
  ],
  temperature = 0
)
test_result = completion.choices[0].message.content
print(test_result)

[
    {
        "exposure": "vitamin E",
        "label": "Vitamin E",
        "category": "Vitamin",
        "outcome_phenotype": "Ischemic stroke",
        "association": "no",
        "effect_size": "not found"
    },
    {
        "exposure": "vitamin C",
        "label": "Vitamin C",
        "category": "Vitamin",
        "outcome_phenotype": "Ischemic stroke",
        "association": "no",
        "effect_size": "not found"
    },
    {
        "exposure": "carotenoids",
        "label": "Carotenoids",
        "category": "Vitamin",
        "outcome_phenotype": "Ischemic stroke",
        "association": "no",
        "effect_size": "not found"
    },
    {
        "exposure": "lutein",
        "label": "Lutein",
        "category": "Vitamin",
        "outcome_phenotype": "Ischemic stroke",
        "association": "negative",
        "effect_size": "not found"
    }
]
time: 18.8 s (started: 2024-07-24 15:32:35 +00:00)


In [ ]:
result_df_1_deepseek = compile_extracted_results(input_df, 'input', prompt)


Processing abstracts: 100%|██████████| 76/76 [02:55<00:00,  2.31s/it]

time: 2min 55s (started: 2024-07-24 15:32:54 +00:00)


In [ ]:
print(result_df_1_deepseek.shape)
left_df = input_df[~input_df['pmid'].isin(result_df_1_deepseek['pmid'])]
left_df

(268, 7)


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 26.2 ms (started: 2024-07-24 15:37:21 +00:00)


In [ ]:
result_df_1_deepseek.to_csv('./data/result_df_1_deepseek.csv')

time: 9.77 ms (started: 2024-07-24 15:37:30 +00:00)


In [ ]:
result_df_1_deepseek['pmid'].nunique()

76

time: 2.49 ms (started: 2024-07-24 15:37:31 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_deepseek.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 10.2 ms (started: 2024-07-25 02:59:12 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

BERTScore for NER (Exposure): (0.6333380341529846, 0.6407017111778259, 0.6333262920379639)
BERTScore for NER (Outcome Phenotype): (0.8409867286682129, 0.8605402112007141, 0.8469316959381104)
time: 26.4 s (started: 2024-07-25 01:58:54 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:44<00:00,  3.75s/it]

BERTScore for NER (Exposure): (0.6538823247446042, 0.666648078219671, 0.6551849426897732)
BERTScore for NER (Outcome Phenotype): (0.8856035741536241, 0.8863408177306777, 0.8824586648690073)
time: 9min 25s (started: 2024-07-25 01:59:21 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [09:10<00:00,  7.24s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.7
Precision: 0.6776218323586743
Recall: 0.7111111111111111
F1-score: 0.6737182496218641
Falsely predicted associations:
|    |     pmid | exposure_gold                                      | exposure_pred                               | outcome_phenotype_gold    | outcome_phenotype_pred    | association_gold   | association_pred   |
|---:|---------:|:---------------------------------------------------|:--------------------------------------------|:--------------------------|:--------------------------|:-------------------|:-------------------|
| 21 | 14980987 | cereal fiber                                       | cereal fiber                                | CHD                       | CHD                       | no                 | negative           |
| 22 | 14980987 | cereal fiber                                       | cereal fiber                                | CHD                       | CHD       

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 12.8 ms (started: 2024-07-31 01:51:22 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

  0%|          | 0/76 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

  4%|▍         | 3/76 [00:16<06:38,  5.45s/it]


KeyboardInterrupt: 

time: 16min 36s (started: 2024-07-31 01:34:43 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:53<00:00,  3.87s/it]

BERTScore for NER (Exposure): (0.7702706836355038, 0.7668738512847668, 0.7625251811857406)
BERTScore for NER (outcome_phenotype): (0.8876344355496955, 0.883428785078069, 0.8810162300125082)
time: 14min 56s (started: 2024-07-31 01:51:28 +00:00)


### GPT 4 - mini

In [ ]:
MODEL="gpt-4o-mini"
#openai
client = OpenAI(api_key='YOUR_API_KEY')

time: 52.1 ms (started: 2024-07-24 15:49:02 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "n-3 fatty acids",
    "label": "n-3 fatty acids",
    "category": "Lipid",
    "outcome_phenotype": "CHD",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "fish intake",
    "label": "Fish intake",
    "category": "Other",
    "outcome_phenotype": "CHD",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "fish intake",
    "label": "Fish intake",
    "category": "Other",
    "outcome_phenotype": "CVD",
    "association": "no",
    "effect_size": "not found"
}]
time: 3.53 s (started: 2024-07-24 15:52:54 +00:00)


In [ ]:
result_df_1_mini= compile_extracted_results(input_df, 'input', prompt)


Processing abstracts: 100%|██████████| 76/76 [00:41<00:00,  1.85it/s]

time: 41.1 s (started: 2024-07-24 15:53:01 +00:00)


In [ ]:
print(result_df_1_mini.shape)
left_df = input_df[~input_df['pmid'].isin(result_df_1_mini['pmid'])]
left_df

(296, 7)


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input
23,24980133,2014.0,Jun,2014.0,6.0,"['Journal Article', 'Research Support, N.I.H.,...",β2-Microglobulin and cystatin C may have advan...,"β2-microglobulin, cystatin C, and creatinine a...",β2-Microglobulin and cystatin C may have advan...,NaN,We conducted nested case-control studies among...,"In pooled analyses, β2-microglobulin was assoc...",We conducted nested case-control studies among...,"In pooled analyses, β2-microglobulin was assoc...",NaN,"['Aged', 'Biomarkers/blood', 'Case-Control Stu...","['Creatinine/blood*', 'Cystatin C/blood*', 'In...","Title: β2-microglobulin, cystatin C, and creat..."


time: 27.4 ms (started: 2024-07-24 15:53:42 +00:00)


In [ ]:
result_df_1_mini.to_csv('./data/result_df_1_mini.csv')

time: 7.19 ms (started: 2024-07-24 15:53:42 +00:00)


In [ ]:
print(result_df_1_mini['pmid'].nunique())

75
time: 499 µs (started: 2024-07-24 15:53:42 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_mini.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['negative' 'no' 'positive']
time: 12.8 ms (started: 2024-07-25 02:59:00 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6182305812835693, 0.6295024752616882, 0.6203271150588989)
BERTScore for NER (Outcome Phenotype): (0.7144513130187988, 0.6963361501693726, 0.7013344168663025)
time: 9.52 s (started: 2024-07-25 02:18:05 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 75/75 [04:20<00:00,  3.48s/it]

BERTScore for NER (Exposure): (0.6308265404403209, 0.6440516890088717, 0.6325437025725842)
BERTScore for NER (Outcome Phenotype): (0.7939646633466085, 0.7695717727641265, 0.7774427271882693)
time: 8min 43s (started: 2024-07-25 02:18:15 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 75/75 [10:22<00:00,  8.30s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.72
Precision: 0.7192251461988305
Recall: 0.7425925925925926
F1-score: 0.7266074520976482
Falsely predicted associations:
|    |     pmid | exposure_gold                   | exposure_pred                   | outcome_phenotype_gold   | outcome_phenotype_pred   | association_gold   | association_pred   |
|---:|---------:|:--------------------------------|:--------------------------------|:-------------------------|:-------------------------|:-------------------|:-------------------|
| 14 | 14963055 | supplemental magnesium          | supplemental magnesium intake   | CHD                      | CHD                      | negative           | no                 |
| 15 | 14980987 | cereal fiber                    | cereal fiber                    | CHD                      | CHD                      | no                 | negative           |
| 16 | 14980987 | cereal fiber                    | cereal fiber       

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_mini.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 75
75 75
['no' 'negative' 'positive'] ['negative' 'no' 'positive']
time: 648 ms (started: 2024-07-31 02:06:25 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 75/75 [04:47<00:00,  3.83s/it]

BERTScore for NER (Exposure): (0.7555661533260718, 0.7545452190679498, 0.7494547421229072)
BERTScore for NER (outcome_phenotype): (0.7627843021545359, 0.7850046390166847, 0.7682491231669661)
time: 14min 39s (started: 2024-07-31 02:06:25 +00:00)


### chatglm

In [ ]:
MODEL="glm-4-airx"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://open.bigmodel.cn/api/paas/v4/")

time: 69.8 ms (started: 2024-07-25 03:20:11 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "dietary intake of n-3 fatty acids",
    "label": "n-3 Fatty Acids",
    "category": "Lipid",
    "outcome_phenotype": "CHD",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "fish intake",
    "label": "Fish Intake",
    "category": "Food/Diet",
    "outcome_phenotype": "CHD",
    "association": "no",
    "effect_size": "not found"
}]
time: 3.04 s (started: 2024-07-25 03:20:12 +00:00)


In [ ]:
result_df_1_glm= compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 76/76 [00:34<00:00,  2.20it/s]

time: 34.6 s (started: 2024-07-25 03:21:19 +00:00)


In [ ]:
print(result_df_1_glm.shape)
print(result_df_1_glm['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_1_glm['pmid'])]
left_df

(279, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 39.4 ms (started: 2024-07-25 03:22:14 +00:00)


In [ ]:
result_df_1_glm.to_csv('./data/result_df_1_glm.csv')

time: 12.4 ms (started: 2024-07-25 03:23:23 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_glm.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 11.4 ms (started: 2024-07-25 03:23:40 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6310924291610718, 0.6584378480911255, 0.640836238861084)
BERTScore for NER (Outcome Phenotype): (0.756875216960907, 0.7422759532928467, 0.7464870810508728)
time: 10.2 s (started: 2024-07-25 03:23:47 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:08<00:00,  3.26s/it]

BERTScore for NER (Exposure): (0.6323002404778412, 0.6641325426140898, 0.6432615864139638)
BERTScore for NER (Outcome Phenotype): (0.8111063636055118, 0.7937496129031244, 0.799049150394766)
time: 8min 24s (started: 2024-07-25 03:23:57 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [08:14<00:00,  6.51s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.8026315789473685
Precision: 0.8101345189238486
Recall: 0.7903703703703705
F1-score: 0.7853954409797262
Falsely predicted associations:
|    |     pmid | exposure_gold                                           | exposure_pred                                  | outcome_phenotype_gold   | outcome_phenotype_pred   | association_gold   | association_pred   |
|---:|---------:|:--------------------------------------------------------|:-----------------------------------------------|:-------------------------|:-------------------------|:-------------------|:-------------------|
| 11 | 10584044 | short- to medium-chain saturated fatty acids (4:0-10:0) | longer-chain saturated fatty acids (12:0-18:0) | CHD                      | CHD                      | no                 | positive           |
| 21 | 14980987 | cereal fiber                                            | cereal fiber                                  

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_glm.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 726 ms (started: 2024-07-31 02:21:05 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:51<00:00,  3.83s/it]

BERTScore for NER (Exposure): (0.7825742028462581, 0.7624049544525452, 0.7673589375729745)
BERTScore for NER (outcome_phenotype): (0.797014305645481, 0.8205913422589607, 0.8037861801208334)
time: 14min 57s (started: 2024-07-31 02:21:05 +00:00)


### Qwen

In [ ]:
MODEL="qwen-plus"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

time: 56.3 ms (started: 2024-07-25 03:49:05 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "marine n-3 fatty acids",
    "label": "Marine n-3 fatty acids",
    "category": "Lipid",
    "outcome_phenotype": "coronary disease",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "fish intake",
    "label": "Fish intake",
    "category": "Food/Diet",
    "outcome_phenotype": "coronary disease",
    "association": "no",
    "effect_size": "not found"
},
{
    "exposure": "fish intake",
    "label": "Fish intake",
    "category": "Food/Diet",
    "outcome_phenotype": "death due to coronary disease",
    "association": "no",
    "effect_size": "not found"
}]
time: 12.5 s (started: 2024-07-25 03:49:11 +00:00)


In [ ]:
result_df_1_qwen= compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 76/76 [02:04<00:00,  1.63s/it]

time: 2min 4s (started: 2024-07-25 03:49:56 +00:00)


In [ ]:
print(result_df_1_qwen.shape)
print(result_df_1_qwen['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_1_qwen['pmid'])]
left_df

(282, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 27.8 ms (started: 2024-07-25 03:52:00 +00:00)


In [ ]:
result_df_1_qwen.to_csv('./data/result_df_1_qwen.csv')

time: 6.26 ms (started: 2024-07-25 03:52:00 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_qwen.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['negative' 'no' 'positive']
time: 10.8 ms (started: 2024-07-25 03:52:33 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6437532305717468, 0.652665376663208, 0.6447054743766785)
BERTScore for NER (Outcome Phenotype): (0.7803532481193542, 0.7654927372932434, 0.7700745463371277)
time: 9.01 s (started: 2024-07-25 03:52:33 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:01<00:00,  3.18s/it]

BERTScore for NER (Exposure): (0.6640411192355188, 0.6667078652076031, 0.6610310525192242)
BERTScore for NER (Outcome Phenotype): (0.8186437444467294, 0.8003529184742978, 0.8061754173158031)
time: 8min (started: 2024-07-25 03:52:42 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [08:03<00:00,  6.37s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.75
Precision: 0.7093582887700535
Recall: 0.7533094812164579
F1-score: 0.7158208122063542
Falsely predicted associations:
|    |     pmid | exposure_gold                                           | exposure_pred                                  | outcome_phenotype_gold    | outcome_phenotype_pred    | association_gold   | association_pred   |
|---:|---------:|:--------------------------------------------------------|:-----------------------------------------------|:--------------------------|:--------------------------|:-------------------|:-------------------|
| 13 | 10584044 | short- to medium-chain saturated fatty acids (4:0-10:0) | longer-chain saturated fatty acids (12:0-18:0) | CHD                       | CHD                       | no                 | positive           |
| 22 | 14980987 | cereal fiber                                            | cereal fiber                                   | CHD  

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_1_qwen.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['negative' 'no' 'positive']
time: 640 ms (started: 2024-07-31 02:36:03 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:49<00:00,  3.82s/it]

BERTScore for NER (Exposure): (0.7657184592806376, 0.7648727514040776, 0.7586274292988655)
BERTScore for NER (outcome_phenotype): (0.8232215708716119, 0.8406970519968804, 0.8284805343347661)
time: 14min 51s (started: 2024-07-31 02:36:03 +00:00)


## prompt 2

In [ ]:
# Define the prompt for step 1
prompt_step1 = """
From the following given title and abstract, extract all studied exposures and cardiovascular-related outcomes separately. Ignore any other information.

Extract the following information:

1. "exposures": a list of all studied exposures
2. "outcomes": a list of all studied cardiovascular-related outcomes. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.

Return the extracted information in a JSON dictionary.

Example input:
""
Title: "Dietary fat and risk of coronary heart disease in men: cohort follow up study in the United States"

Abstract: "Objective: To examine the association between fat intake and the incidence of coronary heart disease in men of middle age and older. Design: Cohort questionnaire study of men followed up for six years from 1986. Setting: The health professionals follow up study in the United States. Subjects: 43 757 health professionals aged 40 to 75 years free of diagnosed cardiovascular disease or diabetes in 1986. Main outcome measure: Incidence of acute myocardial infarction or coronary death. Results: During follow up 734 coronary events were documented, including 505 non-fatal myocardial infarctions and 229 deaths. After age and several coronary risk factors were controlled for significant positive associations were observed between intake of saturated fat and risk of coronary disease. For men in the top versus the lowest fifth of saturated fat intake (median = 14.8% v 5.7% of energy) the multivariate relative risk for myocardial infarction was 1.22 (95% confidence interval 0.96 to 1.56) and for fatal coronary heart disease was 2.21 (1.38 to 3.54). After adjustment for intake of fibre the risks were 0.96 (0.73 to 1.27) and 1.72 (1.01 to 2.90), respectively. Positive associations between intake of cholesterol and risk of coronary heart disease were similarly attenuated after adjustment for fibre intake. Intake of linolenic acid was inversely associated with risk of myocardial infarction; this association became significant only after adjustment for non-dietary risk factors and was strengthened after adjustment for total fat intake (relative risk 0.41 for a 1% increase in energy, P for trend < 0.01)."
Conclusions: These data do not support the strong association between intake of saturated fat and risk of coronary heart disease suggested by international comparisons. They are compatible, however, with the hypotheses that saturated fat and cholesterol intakes affect the risk of coronary heart disease as predicted by their effects on blood cholesterol concentration. They also support a specific preventive effect of linolenic acid intake.
""

Example output:
{
    "exposures": ["fat intake", "saturated fat", "cholesterol", "linolenic acid"],
    "outcomes": ["CHD", "MI"]
}

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ```in your reponse
"""

prompt_step2_template = """
From the following given title and abstract, extract the associations between the given dietary exposures and cardiovascular complications, such as coronary heart disease, stroke, and hypertension.

Use the provided entities as references.

fyi: CHD refers to non-fatal MI and fatal coronary heart disease; also, coronary artery disease (CAD) refers to non-fatal MI and fatal coronary artery disease. Cardiovascular disease (CVD) is defined as a composite of coronary artery disease and non-fatal or fatal stroke.

Extract the following information:

1. "exposure": the studied exposure
2. "label": standardized exposure concept
3. "category": the category of exposure based on 9 categories: 'Food/Diet', 'Lipid', 'Mineral', 'Vitamin', 'Carb', 'Alcohol', 'Fiber', 'Other', 'Protein'
4. "outcome_phenotype": the studied outcome, which is a cardiovascular-related disease such as CHD, stroke, etc. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.
5. "association": the association between exposure and outcome, which is either positive, negative, or no association
6. "effect_size": the effect size of the association. Only return for positive and negative associations. Return "not found" for no association (output number, do not output percentage)

Return the extracted information in a list of flat JSON dictionaries of each result.

Example input entities:
{{
    "exposures": ["fat intake", "saturated fat", "cholesterol", "linolenic acid"],
    "outcomes": ["CHD", "MI"]
}}

Example input title and abstract:
""
Title: "Dietary fat and risk of coronary heart disease in men: cohort follow up study in the United States"

Abstract: "Objective: To examine the association between fat intake and the incidence of coronary heart disease in men of middle age and older. Design: Cohort questionnaire study of men followed up for six years from 1986. Setting: The health professionals follow up study in the United States. Subjects: 43 757 health professionals aged 40 to 75 years free of diagnosed cardiovascular disease or diabetes in 1986. Main outcome measure: Incidence of acute myocardial infarction or coronary death. Results: During follow up 734 coronary events were documented, including 505 non-fatal myocardial infarctions and 229 deaths. After age and several coronary risk factors were controlled for significant positive associations were observed between intake of saturated fat and risk of coronary disease. For men in the top versus the lowest fifth of saturated fat intake (median = 14.8% v 5.7% of energy) the multivariate relative risk for myocardial infarction was 1.22 (95% confidence interval 0.96 to 1.56) and for fatal coronary heart disease was 2.21 (1.38 to 3.54). After adjustment for intake of fibre the risks were 0.96 (0.73 to 1.27) and 1.72 (1.01 to 2.90), respectively. Positive associations between intake of cholesterol and risk of coronary heart disease were similarly attenuated after adjustment for fibre intake. Intake of linolenic acid was inversely associated with risk of myocardial infarction; this association became significant only after adjustment for non-dietary risk factors and was strengthened after adjustment for total fat intake (relative risk 0.41 for a 1% increase in energy, P for trend < 0.01)."
Conclusions: These data do not support the strong association between intake of saturated fat and risk of coronary heart disease suggested by international comparisons. They are compatible, however, with the hypotheses that saturated fat and cholesterol intakes affect the risk of coronary heart disease as predicted by their effects on blood cholesterol concentration. They also support a specific preventive effect of linolenic acid intake.
""

Example output:
[
    {{"exposure": "linolenic acid", "label": "Linolenic acid", "category": "Lipid", "outcome_phenotype": "MI", "association": "negative", "effect_size": "0.41"}},
    {{"exposure": "saturated fat", "label": "SFA", "category": "Lipid", "outcome_phenotype": "CHD", "association": "positive", "effect_size": "2.21"}},
    {{"exposure": "saturated fat", "label": "SFA", "category": "Lipid", "outcome_phenotype": "MI", "association": "positive", "effect_size": "1.22"}},
    {{"exposure": "saturated fat adjusted for fiber", "label": "SFA adjusted for fiber", "category": "Lipid", "outcome_phenotype": "CHD", "association": "no", "effect_size": "not found"}},
    {{"exposure": "saturated fat adjusted for fiber", "label": "SFA adjusted for fiber", "category": "Lipid", "outcome_phenotype": "MI", "association": "no", "effect_size": "not found"}}
]

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your reponse
"""

time: 606 µs (started: 2024-07-25 06:23:23 +00:00)


In [ ]:
import openai
import pandas as pd
from tqdm import tqdm
from ast import literal_eval
import threading
from queue import Queue
import json

# Function to perform the two-step extraction
def two_step_extraction(text):
    # Step 1: Extract exposures and outcomes
    completion_step1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
            {"role": "user", "content": prompt_step1},
            {"role": "user", "content": text}
        ]
    )
    entities_result = completion_step1.choices[0].message.content
    entities_result = entities_result.strip().removeprefix("```json").removesuffix("```").strip()
    entities_dict = json.loads(entities_result)
    entities_json = json.dumps(entities_dict, indent=2)

    # Prepare step 2 prompt
    prompt_step2 = prompt_step2_template.format(entities=entities_json)

    # Step 2: Extract associations using the entities
    completion_step2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
            {"role": "user", "content": prompt_step2},
            {"role": "user", "content": f"extracted entities: {entities_json}"},
            {"role": "user", "content": f"title and abstract: {text}"}
        ]
    )
    associations_result = completion_step2.choices[0].message.content

    return associations_result

# Worker function to process each row
def worker(input_queue, output_queue, text_column, progress_bar):
    while True:
        index, row = input_queue.get()
        if index is None:
            break
        try:
            output = two_step_extraction(row[text_column])
            extracted_results = literal_eval(output)
            for res in extracted_results:
                res['pmid'] = row['pmid']
                output_queue.put(res)
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            print(f"Input text: {row[text_column][:500]}...")  # Print the first 500 characters of the text
            print(f"Output: {output}")  # Print the output if available
        finally:
            input_queue.task_done()
            progress_bar.update(1)

# Main function to compile extracted results
def compile_extracted_results(input_df, text_column):
    results = []
    input_queue = Queue()
    output_queue = Queue()

    # Progress bar for processing
    progress_bar = tqdm(total=input_df.shape[0], desc="Processing abstracts")

    # Start worker threads
    for _ in range(8):
        threading.Thread(target=worker, args=(input_queue, output_queue, text_column, progress_bar), daemon=True).start()

    # Enqueue rows for processing
    for index, row in input_df.iterrows():
        input_queue.put((index, row))

    # Wait for all tasks to be processed
    input_queue.join()

    # Collect results from output queue
    while not output_queue.empty():
        results.append(output_queue.get())

    # Close progress bar
    progress_bar.close()

    results_df = pd.json_normalize(results)
    return results_df

time: 1.93 ms (started: 2024-07-25 06:26:28 +00:00)


### deepseek

In [ ]:
MODEL="deepseek-chat"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.deepseek.com")

time: 51.5 ms (started: 2024-07-25 05:21:11 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_3}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["vitamin E", "vitamin C", "carotenoids", "lutein"],
    "outcomes": ["stroke", "ischemic stroke", "hemorrhagic stroke"]
}
Associations extracted: [
    {"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "lutein", "label": "Lutein", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "negative", "effect_size": "not found"}
]
time: 13.5 s (started: 2024-07-25 05:21:13 +00:00)


In [ ]:
result_df_2_deepseek = compile_extracted_results(input_df, 'input')

Processing abstracts: 100%|██████████| 76/76 [02:46<00:00,  2.20s/it]

time: 2min 46s (started: 2024-07-25 05:29:50 +00:00)


In [ ]:
print(result_df_2_deepseek.shape)
print(result_df_2_deepseek['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_2_deepseek['pmid'])]
left_df

(313, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 29.4 ms (started: 2024-07-25 05:44:48 +00:00)


In [ ]:
result_df_2_deepseek.to_csv('./data/result_df_2_deepseek.csv')

time: 8.62 ms (started: 2024-07-25 05:45:02 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_deepseek.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 11.2 ms (started: 2024-09-01 23:42:20 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

BERTScore for NER (Exposure): (0.6650554537773132, 0.6611852049827576, 0.6595773696899414)
BERTScore for NER (Outcome Phenotype): (0.7617071270942688, 0.7461860179901123, 0.75071781873703)
time: 28.9 s (started: 2024-09-01 22:03:02 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [03:56<00:00,  3.12s/it]

BERTScore for NER (Exposure): (0.6500712586566806, 0.6541901482759338, 0.647788098817201)
BERTScore for NER (Outcome Phenotype): (0.8499687496376666, 0.824774842415201, 0.8330140049128156)
time: 7min 56s (started: 2024-07-25 05:45:39 +00:00)


##### relationship

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [11:15<00:00,  8.88s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.7950819672131147
Precision: 0.737675566680573
Recall: 0.8265750855125235
F1-score: 0.7548148148148148
Falsely predicted associations:
|     |     pmid | exposure_gold                      | exposure_pred                      | outcome_phenotype_gold    | outcome_phenotype_pred    | association_gold   | association_pred   |
|----:|---------:|:-----------------------------------|:-----------------------------------|:--------------------------|:--------------------------|:-------------------|:-------------------|
|  20 | 14980987 | cereal fiber                       | cereal fiber                       | CHD                       | CHD                       | no                 | negative           |
|  21 | 14980987 | cereal fiber                       | cereal fiber                       | CHD                       | CHD                       | no                 | negative           |
|  26 | 15585762 | sup

##### error_analysis

In [ ]:
input_df.columns

Index(['pmid', 'pub_year', 'pub_month', 'epub_year', 'epub_month', 'pub_types',
       'full_abs', 'article_title', 'background', 'method', 'result',
       'conclusion', 'label_result', 'label_conclusion', '# of comments',
       'all_mesh_terms', 'major_mesh_terms', 'input'],
      dtype='object')

time: 3.05 ms (started: 2024-09-01 23:43:20 +00:00)


In [ ]:
error_df = pd.merge(input_df[['pmid','input']].astype(str),falsely_predicted.astype(str),on='pmid',how='right')
error_df.to_excel('./data/result_df_2_deepseek_error.xlsx')
error_df

,pmid,input,exposure_gold,exposure_pred,outcome_phenotype_gold,outcome_phenotype_pred,association_gold,association_pred
0,14980987,Title: Dietary fiber and risk of coronary hear...,cereal fiber,cereal fiber,CHD,CHD,no,negative
1,14980987,Title: Dietary fiber and risk of coronary hear...,cereal fiber,cereal fiber,CHD,CHD,no,negative
2,15585762,Title: Antioxidant vitamins and coronary heart...,supplemental vitamin C,vitamin C,CHD,CHD,negative,no
3,15585762,Title: Antioxidant vitamins and coronary heart...,supplemental vitamin C,supplemental vitamin E,CHD,CHD,negative,no
4,15585762,Title: Antioxidant vitamins and coronary heart...,supplemental vitamin E,supplemental vitamin C,CHD,CHD,no,negative
5,20881068,Title: Dietary protein and risk of ischemic he...,vegetable protein,vegetable protein,Ischemic heart disease,Ischemic Heart Disease,negative,no
6,24819748,"Title: Plasma homocysteine, dietary B vitamins...",homocysteine,plasma homocysteine,Peripheral artery disease,Peripheral artery disease,no,positive
7,24819748,"Title: Plasma homocysteine, dietary B vitamins...",homocysteine,plasma homocysteine,Peripheral artery disease,Peripheral artery disease,no,positive
8,24980133,"Title: β2-microglobulin, cystatin C, and creat...",b2-microglobulin,β2-microglobulin,Peripheral artery disease,Peripheral artery disease,no,positive
9,24980133,"Title: β2-microglobulin, cystatin C, and creat...",b2-microglobulin,β2-microglobulin,Peripheral artery disease,Peripheral artery disease,no,positive


time: 305 ms (started: 2024-09-01 23:44:54 +00:00)


#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 18.5 ms (started: 2024-07-31 03:07:58 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

  0%|          | 0/76 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

 76%|███████▋  | 58/76 [11:27<03:02, 10.15s/it]

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

  0%|          | 0/76 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 76/76 [04:30<00:00,  3.56s/it]

BERTScore for NER (Exposure): (0.778878531471277, 0.7902503262918729, 0.7790862676997979)
BERTScore for NER (outcome_phenotype): (0.8674582803344473, 0.882328711925669, 0.870341625619442)
time: 14min 2s (started: 2024-07-31 03:08:04 +00:00)


##### error_analysis

### GPT-4 mini

In [ ]:
MODEL="gpt-4o-mini"
#openai
client = OpenAI(api_key='YOUR_API_KEY')

time: 52.9 ms (started: 2024-07-25 06:01:38 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_3}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["vitamin E", "vitamin C", "carotenoids"],
    "outcomes": ["stroke", "ischemic stroke"]
}
Associations extracted: [
    {"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "carotenoids", "label": "Carotenoids", "category": "Other", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "stroke", "association": "no", "effect_size": "not found"},
    {"exposure": "carotenoids", "la

In [ ]:
result_df_2_mini = compile_extracted_results(input_df, 'input')

Processing abstracts: 100%|██████████| 76/76 [01:19<00:00,  1.04s/it]

time: 1min 19s (started: 2024-07-25 06:01:47 +00:00)


In [ ]:
print(result_df_2_mini.shape)
print(result_df_2_mini['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_2_mini['pmid'])]
left_df

(313, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 29.4 ms (started: 2024-07-25 05:44:48 +00:00)


In [ ]:
result_df_2_mini.to_csv('./data/result_df_2_mini.csv')

time: 7.12 ms (started: 2024-07-25 06:03:06 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_mini.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['negative' 'positive' 'not found' 'no']
time: 9.11 ms (started: 2024-07-25 06:03:06 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6577060222625732, 0.6602085828781128, 0.6554800868034363)
BERTScore for NER (Outcome Phenotype): (0.7674722671508789, 0.7402101755142212, 0.7489444613456726)
time: 8.35 s (started: 2024-07-25 06:03:06 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:00<00:00,  3.16s/it]

BERTScore for NER (Exposure): (0.6454235260914031, 0.6614888358469072, 0.6491755643663438)
BERTScore for NER (Outcome Phenotype): (0.8119124907412028, 0.7765831952228358, 0.788014040965783)
time: 7min 56s (started: 2024-07-25 06:03:14 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [08:01<00:00,  6.34s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.782608695652174
Precision: 0.7608695652173912
Recall: 0.8169934640522877
F1-score: 0.7760269360269362
Falsely predicted associations:
|    |     pmid | exposure_gold                        | exposure_pred                        | outcome_phenotype_gold   | outcome_phenotype_pred   | association_gold   | association_pred   |
|---:|---------:|:-------------------------------------|:-------------------------------------|:-------------------------|:-------------------------|:-------------------|:-------------------|
| 18 | 14980987 | cereal fiber                         | cereal fiber                         | CHD                      | CHD                      | no                 | negative           |
| 19 | 14980987 | cereal fiber                         | cereal fiber                         | CHD                      | CHD                      | no                 | negative           |
| 25 | 15585762 | 

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_mini.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['negative' 'positive' 'not found' 'no']
time: 16.6 ms (started: 2024-07-31 03:45:04 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

  0%|          | 0/76 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 76/76 [04:30<00:00,  3.56s/it]

BERTScore for NER (Exposure): (0.791715085812104, 0.7903798156059705, 0.784010821982072)
BERTScore for NER (outcome_phenotype): (0.8119362111579864, 0.8310381212133042, 0.817285733337098)
time: 14min 2s (started: 2024-07-31 03:45:09 +00:00)


### chatglm

In [ ]:
MODEL="glm-4-airx"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://open.bigmodel.cn/api/paas/v4/")

time: 74.3 ms (started: 2024-07-25 06:26:33 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_3}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["vitamin E", "vitamin C", "carotenoids", "lutein"],
    "outcomes": ["total stroke", "ischemic stroke", "hemorrhagic stroke"]
}
Associations extracted: [
    {{"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "positive", "effect_size": "1.18"}},
    {{"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "positive", "effect_size": "1.03"}},
    {{"exposure": "lutein", "label": "Lutein", "category": "Carotenoids", "outcome_phenotype": "ischemic stroke", "association": "negative", "effect_size": "not found"}}
]
time: 5.18 s (started: 2024-07-25 06:26:33 +00:00)


In [ ]:
result_df_2_glm = compile_extracted_results(input_df, 'input')


Processing abstracts: 100%|██████████| 76/76 [00:54<00:00,  1.40it/s]

time: 54.3 s (started: 2024-07-25 06:27:02 +00:00)


In [ ]:
print(result_df_2_glm.shape)
print(result_df_2_glm['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_2_glm['pmid'])]
left_df

(316, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 26.3 ms (started: 2024-07-25 06:27:56 +00:00)


In [ ]:
result_df_2_glm.to_csv('./data/result_df_2_glm.csv')

time: 7.39 ms (started: 2024-07-25 06:27:56 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_glm.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['positive' 'negative' 'not found' 'no']
time: 9.09 ms (started: 2024-07-25 06:28:04 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6344059109687805, 0.6426664590835571, 0.6348124146461487)
BERTScore for NER (Outcome Phenotype): (0.7444671392440796, 0.7357583045959473, 0.737162172794342)
time: 10.2 s (started: 2024-07-25 06:28:05 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)


100%|██████████| 76/76 [03:07<00:00,  2.47s/it]

100%|██████████| 76/76 [03:03<00:00,  2.42s/it]

BERTScore for NER (Exposure): (0.6238297463737821, 0.6482055509757054, 0.6313270558358023)
BERTScore for NER (Outcome Phenotype): (0.8035003633091324, 0.7931644766542473, 0.7938004826244555)
time: 6min 11s (started: 2024-07-25 06:28:15 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())


100%|██████████| 76/76 [06:06<00:00,  4.82s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.8571428571428571
Precision: 0.8699494949494949
Recall: 0.8784679089026914
F1-score: 0.8599470116503176
Falsely predicted associations:
|    |     pmid | exposure_gold                                           | exposure_pred                                  | outcome_phenotype_gold   | outcome_phenotype_pred   | association_gold   | association_pred   |
|---:|---------:|:--------------------------------------------------------|:-----------------------------------------------|:-------------------------|:-------------------------|:-------------------|:-------------------|
| 10 | 10584044 | short- to medium-chain saturated fatty acids (4:0-10:0) | longer-chain saturated fatty acids (12:0-18:0) | CHD                      | CHD                      | no                 | positive           |
| 16 | 14980987 | cereal fiber                                            | cereal fiber intake                           

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_glm.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['positive' 'negative' 'not found' 'no']
time: 12.6 ms (started: 2024-07-31 03:59:12 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:31<00:00,  3.58s/it]

BERTScore for NER (Exposure): (0.7705308323487258, 0.7583848480612804, 0.7577650346626074)
BERTScore for NER (outcome_phenotype): (0.8546372089297214, 0.8515961883549995, 0.8492200666285575)
time: 13min 54s (started: 2024-07-31 03:59:12 +00:00)


### qwen

In [ ]:
MODEL="qwen-plus"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

time: 50.1 ms (started: 2024-07-25 06:41:43 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_3}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["vitamin E", "vitamin C", "carotenoids", "lutein"],
    "outcomes": ["stroke", "ischemic stroke", "hemorrhagic stroke"]
}
Associations extracted: [
    {{"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"}},
    {{"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"}},
    {{"exposure": "lutein", "label": "Lutein", "category": "Carotenoids", "outcome_phenotype": "ischemic stroke", "association": "negative", "effect_size": "not found"}},
    {{"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "stroke", "association": "no", "effect_size": "not found"}},
    {{"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "stroke", "association": "no", "effect_size": "not f

In [ ]:
result_df_2_qwen = compile_extracted_results(input_df, 'input')


Processing abstracts: 100%|██████████| 76/76 [02:22<00:00,  1.88s/it]

time: 2min 22s (started: 2024-07-25 06:42:04 +00:00)


In [ ]:
print(result_df_2_qwen.shape)
print(result_df_2_qwen['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(result_df_2_qwen['pmid'])]
left_df

(313, 7)
76


,pmid,pub_year,pub_month,epub_year,epub_month,pub_types,full_abs,article_title,background,method,result,conclusion,label_result,label_conclusion,# of comments,all_mesh_terms,major_mesh_terms,input


time: 27.1 ms (started: 2024-07-25 06:44:27 +00:00)


In [ ]:
result_df_2_qwen.to_csv('./data/result_df_2_qwen.csv')

time: 13.6 ms (started: 2024-07-25 06:44:27 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_qwen.csv')

print(gold_df['association'].unique(),pred_df['association'].unique())

['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 9 ms (started: 2024-07-25 06:44:27 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Calculate BERTScore
def calculate_bertscore(gold, pred, column):
    P, R, F1 = score(pred[column + '_pred'].tolist(), gold[column + '_gold'].tolist(), lang='en', model_type='microsoft/deberta-xlarge-mnli')
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Calculate BERTScore for NER
bertscore_ner_exposure = calculate_bertscore(merged_df, merged_df, 'exposure')
bertscore_ner_outcome = calculate_bertscore(merged_df, merged_df, 'outcome_phenotype')

# Output results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome)

BERTScore for NER (Exposure): (0.6338250637054443, 0.6474705338478088, 0.6371181011199951)
BERTScore for NER (Outcome Phenotype): (0.7507408261299133, 0.7353547811508179, 0.7393329739570618)
time: 6.94 s (started: 2024-07-25 06:44:27 +00:00)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)


100%|██████████| 76/76 [03:04<00:00,  2.43s/it]

100%|██████████| 76/76 [03:05<00:00,  2.44s/it]

BERTScore for NER (Exposure): (0.6265066258707329, 0.6482948336358133, 0.6328231457522825)
BERTScore for NER (Outcome Phenotype): (0.8257530302201447, 0.8104190719559005, 0.8138048737064788)
time: 6min 10s (started: 2024-07-25 06:44:34 +00:00)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())


100%|██████████| 76/76 [06:10<00:00,  4.87s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.7865168539325843
Precision: 0.7526532607177768
Recall: 0.8093681917211328
F1-score: 0.7733074042062805
Falsely predicted associations:
|    |     pmid | exposure_gold                                           | exposure_pred                                       | outcome_phenotype_gold    | outcome_phenotype_pred    | association_gold   | association_pred   |
|---:|---------:|:--------------------------------------------------------|:----------------------------------------------------|:--------------------------|:--------------------------|:-------------------|:-------------------|
| 10 | 10584044 | short- to medium-chain saturated fatty acids (4:0-10:0) | longer-chain saturated fatty acids (12:0-18:0)      | CHD                       | CHD                       | no                 | positive           |
| 17 | 14963055 | supplemental magnesium                                  | supplemental magnesium in

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/nc_vali_df.csv')
pred_df = pd.read_csv('./data/result_df_2_qwen.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['association'].unique(),pred_df['association'].unique())

76 76
76 76
['no' 'negative' 'positive'] ['positive' 'negative' 'no']
time: 1.91 s (started: 2024-07-31 04:13:07 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [04:35<00:00,  3.63s/it]

BERTScore for NER (Exposure): (0.7550765574742586, 0.744198820892817, 0.7435185675246593)
BERTScore for NER (outcome_phenotype): (0.8808061918838227, 0.8833875602230112, 0.8806946011299782)
time: 13min 57s (started: 2024-07-31 04:13:09 +00:00)


# AMIA validation dataset

In [ ]:
amia_vali_df = pd.read_csv('./data/amia_vali_df.csv').drop(columns=['Unnamed: 0'])
amia_vali_df['pmid'] = amia_vali_df['pmid'].astype(str)
print(amia_vali_df.columns)
amia_vali_df = amia_vali_df.dropna(subset=['pmid', 'full_abs', 'article_title', 'direction'])
print(amia_vali_df['pmid'].nunique())
amia_vali_df['direction'] = amia_vali_df['direction'].apply(
    lambda x: x.replace('unchanged', 'not found').replace('indifferent', 'not found')
)
print(amia_vali_df['direction'].value_counts())


Index(['result_num', 'expert', 'pmid', 'article_title', 'article_title_ch',
       'full_abs', 'full_abs_ch', 'exposure', 'outcome', 'significance',
       'direction', 'design', 'included_studies', 'number_of_participants',
       'population_main_condition', 'comparator', 'notes'],
      dtype='object')
90
direction
decrease     148
increase     101
not found     60
Name: count, dtype: int64
time: 4.54 s (started: 2024-09-02 05:08:59 +00:00)


In [ ]:
amia_vali_df

,pmid,full_abs,exposure,outcome,significance,direction,population_main_condition,comparator
1,2856757,The effect on blood pressure of an ovo-lacto-v...,ovo-lacto-vegetarian (OLV) diet,systolic blood pressure,positive,decrease,mild untreated hypertensive subjects,usual diet
2,3178506,"The effect of fish oil on lipids, coagulation ...",fish oil,triglycerides,positive,decrease,stable angina pectoris,n-6 polyunsaturated fatty acids
3,3178506,"The effect of fish oil on lipids, coagulation ...",fish oil,very low density lipoproteins,positive,decrease,stable angina pectoris,n-6 polyunsaturated fatty acids
4,3178506,"The effect of fish oil on lipids, coagulation ...",fish oil,total cholesterol,negative,increase,stable angina pectoris,n-6 polyunsaturated fatty acids
5,3178506,"The effect of fish oil on lipids, coagulation ...",fish oil,fibrinolytic activity at rest,negative,decrease,stable angina pectoris,n-6 polyunsaturated fatty acids
...,...,...,...,...,...,...,...,...
318,37354762,Ultraprocessed foods (UPF) consumption plays a...,ultraprocessed food consumption,diastolic blood pressure,positive,not found,6-year-old children,NaN
319,37488260,The effect of time-restricted eating (TRE) has...,Time-restricted eating with calorie restriction,body weight,positive,decrease,NaN,calorie restriction alone
320,37488260,The effect of time-restricted eating (TRE) has...,Time-restricted eating with calorie restriction,fat mass,positive,decrease,NaN,calorie restriction alone
321,37488260,The effect of time-restricted eating (TRE) has...,Time-restricted eating with calorie restriction,waist circumference,positive,decrease,NaN,calorie restriction alone


time: 33.3 ms (started: 2024-09-02 05:09:04 +00:00)


In [ ]:
amia_input_df = pd.read_csv('./data/sdoh_pmids_100.csv').drop(columns=['Unnamed: 0'])
#('./data/nc_pmids_result.csv').drop(columns=['Unnamed: 0'])

print(amia_input_df.shape,amia_input_df.columns)
amia_input_df['input'] = "Title: " + amia_input_df['article_title'] + '. Abstract: ' + amia_input_df['full_abs']
amia_input_df['pmid'] = amia_input_df['pmid'].astype(int).astype(str)
#nc_input_df  = nc_input_df.dropna(subset=['input'])
print(amia_input_df.shape,amia_input_df.columns)

amia_input_df['pmid'] = amia_input_df['pmid'].astype(str)

input_df = amia_input_df[amia_input_df['pmid'].isin(amia_vali_df['pmid'].tolist())]

print(input_df['pmid'].nunique())

(100, 3) Index(['pmid', 'article_title', 'full_abs'], dtype='object')
(100, 4) Index(['pmid', 'article_title', 'full_abs', 'input'], dtype='object')
90
time: 921 ms (started: 2024-09-02 05:09:06 +00:00)


In [ ]:
test_input_1 = """ Title: Effects of introducing linseed in livestock diet on blood fatty acid composition of consumers of animal products. Abstract: Reducing the ratio between essential fatty acids: C18:2 n-6/C18:3 n-3 down to 5 is recommended by Nutritional Guidelines. We studied the fatty acid (FA) changes in consumers' plasma following changes in livestock diet. First, a zootechnical study introduced 5% of extruded linseed into the diet of livestock to replace other oleaginous ingredients, and on an iso-nutritional values basis. The products from linseed-fed animals contained more n-3 fatty acids (precursor alpha-linolenic and derivatives obtained by elongations and desaturations) than control animal products (issued from animals fed without linseed), and more conjugated linoleic acids (CLA). The n-6/n-3 ratio was reduced by 54% in butter, 60% in meat and 86% in eggs. Following this, a double-blind, randomised, cross-over clinical study involving 75 healthy volunteers compared plasma and erythrocyte FA profiles in consumers of animal products (from livestock fed the linseed diet or from livestock fed standard diet). It showed modifications in the FA composition of the experimental human regimen with more C18:3 n-3 (1.65 vs. 0.75 g/day), and more n-3 derivatives. The C18:2 n-6/C18:3 n-3 ratio decreased (7 vs. 15). In volunteers' plasma, C18:3 n-3 increased in the essay group (0.93 vs. 0.44% of the FA), so did n-3 derivatives and CLA. The n-6/n-3 ratio decreased from 14.3 to 10.2. In erythrocytes, C20:5 n-3 increased in the essay group (0.59 vs. 0.45%) and so did C22:6 n-3. The n-6/n-3 ratio decreased in parallel from 4.2 to 3.8. Without any changes in consumers' eating habits, foodstuffs from animals fed linseed diets induced significant modifications of human plasma and erythrocyte fatty acid composition (comparable to that noted under the 'Cretan' diet) and a sharp increase in CLA."""
test_input_2 = """ Title: Acute and chronic oral magnesium supplementation: effects on endothelial function, exercise capacity, and quality of life in patients with symptomatic heart failure.. Abstract: Endothelial dysfunction is an important pathophysiologic mechanism in the progression of heart failure. The objective of the present study was to determine the effects of acute and chronic oral magnesium supplementation on endothelial function in patients with symptomatic heart failure. Twenty-two symptomatic chronic heart failure patients were randomized to receive 800 mg oral magnesium oxide daily or placebo for 3 months. Data collected included large and small arterial elasticity/compliance, hemodynamic parameters, exercise capacity, and quality-of-life score at baseline, 1 week, and 3 months. Patients who received magnesium had improved small arterial compliance at 3 months from baseline compared with placebo. This study suggests that chronic supplementation with oral magnesium is well tolerated and could improve endothelial function in symptomatic heart failure patients."""
test_input_3 = """ Title: Salt restriction in hypertension--the effect of dietary advice and self monitoring of chloride concentration in urine.

The aims of this present study were firstly to examine whether diet compliance and blood pressure effect could be enhanced by self monitoring with a titrator strip (Quantab 1176) measuring the urine chloride concentration. This was achieved by an open, randomized parallel group design. The study also sought to evaluate the blood pressure effect of a moderately salt restricted diet by using a pre-test-post-test design with a run-in period and controlling for relevant confounding factors (weight, training and alcohol consumption). Furthermore, the study aims were to validate the measurement of chloride concentration in the morning urine by the Quantab titrator strip. We compared Quantab 1176 measurement of chloride concentration in the morning urine with 24 h sodium excretion, determined by the clinical chemical laboratory, was performed. Twenty men and 14 women (mean age 53 years) with essential hypertension (mean: 165/96 mmHg) were observed during a run-in period of 4 weeks before randomization to either dietary advice combined with self monitoring of morning urine chloride concentration for 12 weeks, or dietary advice alone. The reduction in diastolic blood pressure of 6 mmHg was not different in the two groups (between groups p = 0.44). Within group changes of systolic blood pressure were 10 mmHg and 6 mmHg (p = 0.006 and p = 0.04) in the diet plus Quantab group, and the diet only group respectively (between groups p = 0.30). No significant difference in 24 h sodium excretion could be detected between the groups. The morning urine chloride concentration correlated moderately to the 24 h urine sodium excretion (r = 0.66, p < 0.001).(ABSTRACT TRUNCATED AT 250 WORDS)"""

time: 428 µs (started: 2024-09-03 03:18:51 +00:00)


In [ ]:
amia_vali_df.sample(3)

,pmid,exposure,outcome,significance,direction,population_main_condition,comparator
166,25421976,dietary nitrate,clinic BP,positive,decrease,patients with hypertension,nitrate-free beetroot juice
187,26269239,the American almonds group (AA),LDL cholesterol,positive,increase,coronary artery disease (CAD) patients with lo...,no-intervention
317,37354762,ultraprocessed food consumption,systolic blood pressure,positive,not found,6-year-old children,NaN


time: 17.6 ms (started: 2024-07-30 02:40:20 +00:00)


## prompt #1

In [ ]:
prompt = """

From the following given title and abstract, extract the associations between exposures and outcomes.
*More than one association might be studied in a paper. Extract all primary results (one exposure to one outcome pairs)

extract following information:

1. "exposure": the studied exposure
2. "outcome": the studied outcome
3. "significance": only return 'positive' if there exists statistical significance or'negative' if no significance;
4. "direction": return the causal direction of the relationship (strict 3 categories): increase, decrease, or not found. (not found include relationshisp such as indifferent or unchanged )
5. "population_main_condition": disease of studied population
6. "comparator": compared group in relationship

Return "not found " if not found or not applicable.

Return the extracted information in a list of flat JSON dictionaries of each result.

example input:
""
Title: "Short- and long-term black tea consumption reverses endothelial dysfunction in patients with coronary artery disease."

Abstract: "BACKGROUND: Epidemiological studies suggest that tea consumption decreases cardiovascular risk, but the mechanisms of benefit remain undefined. Endothelial dysfunction has been associated with coronary artery disease and increased oxidative stress. Some antioxidants have been shown to reverse endothelial dysfunction, and tea contains antioxidant flavonoids. Methods and Results-- To test the hypothesis that tea consumption will reverse endothelial dysfunction, we randomized 66 patients with proven coronary artery disease to consume black tea and water in a crossover design. Short-term effects were examined 2 hours after consumption of 450 mL tea or water. Long-term effects were examined after consumption of 900 mL tea or water daily for 4 weeks. Vasomotor function of the brachial artery was examined at baseline and after each intervention with vascular ultrasound. Fifty patients completed the protocol and had technically suitable ultrasound measurements. Both short- and long-term tea consumption improved endothelium- dependent flow-mediated dilation of the brachial artery, whereas consumption of water had no effect (P<0.001 by repeated-measures ANOVA). Tea consumption had no effect on endothelium-independent nitroglycerin-induced dilation. An equivalent oral dose of caffeine (200 mg) had no short-term effect on flow-mediated dilation. Plasma flavonoids increased after short- and long-term tea consumption. CONCLUSIONS: Short- and long-term black tea consumption reverses endothelial vasomotor dysfunction in patients with coronary artery disease. This finding may partly explain the association between tea intake and decreased cardiovascular disease events."
example output:
[{
    "exposure": "black tea consumption",
    "outcome": "endothelial vasomotor dysfunction",
    "significance": "positive",
    "direction": "decrease",
    "population_main_condition": "coronary artery disease",
    "comparator": "water"

},
{
    "exposure": "black tea consumption",
    "outcome": "endothelium- dependent flow-mediated dilation of the brachial artery",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "coronary artery disease",
    "comparator": "water"

},
{
    "exposure": "black tea consumption",
    "outcome": "endothelium-independent nitroglycerin-induced dilation",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "coronary artery disease",
    "comparator": "water"

},
{
    "exposure": "black tea consumption",
    "outcome": "Plasma flavonoids",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "coronary artery disease",
    "comparator": "water"

}
]

still return a list of dictionaries with empty values if no association is found or input is irrelevant
Only output in json format, do not include any explanations or additional text in your response.
*Dont output ```json ``` at the beginning nor end
"""


test_prompt = prompt

system_1 = 'You are an evidence-based medicine annotator in social determinants of health (SDoh) and cardiovascular diseases, help me extract strucutured information from free texts of study titles and abstracts"'

time: 444 µs (started: 2024-07-30 02:40:22 +00:00)


In [ ]:
import pandas as pd
from tqdm import tqdm
from ast import literal_eval
import threading
from queue import Queue

# Function to extract primary results
def extract_primary_result(text, prompt):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_1},
            {"role": "user", "content": prompt},
            {"role": "user", "content": text}
        ],
        temperature=0,
        response_format={ "type": "json_object" }
    )
    return completion.choices[0].message.content

# Worker function to process each row
def worker(input_queue, output_queue, text_column, prompt, progress_bar):
    while True:
        index, row = input_queue.get()
        if index is None:
            break
        try:
            output = extract_primary_result(row[text_column], prompt)
            extracted_results = literal_eval(output)
            for res in extracted_results:
                res['pmid'] = row['pmid']
                output_queue.put(res)
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            print(f"Input text: {row[text_column][:500]}...")  # Print the first 500 characters of the text
            print(f"Output: {output}")  # Print the output if available
        finally:
            input_queue.task_done()
            progress_bar.update(1)

# Main function to compile extracted results
def compile_extracted_results(input_df, text_column, prompt):
    results = []
    input_queue = Queue()
    output_queue = Queue()

    # Progress bar for processing
    progress_bar = tqdm(total=input_df.shape[0], desc="Processing abstracts")

    # Start 6 worker threads
    for _ in range(10):
        threading.Thread(target=worker, args=(input_queue, output_queue, text_column, prompt, progress_bar), daemon=True).start()

    # Enqueue rows for processing
    for index, row in input_df.iterrows():
        input_queue.put((index, row))

    # Wait for all tasks to be processed
    input_queue.join()

    # Collect results from output queue
    while not output_queue.empty():
        results.append(output_queue.get())

    # Close progress bar
    progress_bar.close()

    results_df = pd.json_normalize(results)
    return results_df

# Example usage
# Assuming you have a DataFrame `input_df` with the necessary columns and a prompt string `prompt`
# results_df = compile_extracted_results(input_df, "abstract", prompt)
# print(results_df)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<ipython-input-16-5fb8c0443a47>, line 16)

time: 2.42 ms (started: 2024-07-30 02:41:49 +00:00)


In [ ]:
input_df['input'].tolist()[2]

'Title: Acute and chronic oral magnesium supplementation: effects on endothelial function, exercise capacity, and quality of life in patients with symptomatic heart failure.. Abstract: Endothelial dysfunction is an important pathophysiologic mechanism in the progression of heart failure. The objective of the present study was to determine the effects of acute and chronic oral magnesium supplementation on endothelial function in patients with symptomatic heart failure. Twenty-two symptomatic chronic heart failure patients were randomized to receive 800 mg oral magnesium oxide daily or placebo for 3 months. Data collected included large and small arterial elasticity/compliance, hemodynamic parameters, exercise capacity, and quality-of-life score at baseline, 1 week, and 3 months. Patients who received magnesium had improved small arterial compliance at 3 months from baseline compared with placebo. This study suggests that chronic supplementation with oral magnesium is well tolerated and 

time: 3.02 ms (started: 2024-07-30 02:41:50 +00:00)


### deepseek

In [ ]:
MODEL="deepseek-chat"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.deepseek.com")

time: 52.2 ms (started: 2024-07-30 02:41:51 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_3}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "dietary advice combined with self monitoring of morning urine chloride concentration",
    "outcome": "diastolic blood pressure",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "essential hypertension",
    "comparator": "dietary advice alone"
},
{
    "exposure": "dietary advice combined with self monitoring of morning urine chloride concentration",
    "outcome": "systolic blood pressure",
    "significance": "positive",
    "direction": "decrease",
    "population_main_condition": "essential hypertension",
    "comparator": "dietary advice alone"
},
{
    "exposure": "dietary advice combined with self monitoring of morning urine chloride concentration",
    "outcome": "24 h sodium excretion",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "essential hypertension",
    "comparator": "dietary advice alone"
},
{
    "exposure": "morning urine chloride concentration",
    

In [ ]:
amia_result_df_1_deepseek = compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 90/90 [02:36<00:00,  1.73s/it]

time: 2min 36s (started: 2024-07-26 06:01:03 +00:00)


In [ ]:
print(amia_result_df_1_deepseek.shape)
print(amia_result_df_1_deepseek['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_1_deepseek['pmid'])]
left_df

(321, 7)
90


,pmid,article_title,full_abs,input


time: 12.2 ms (started: 2024-07-26 06:03:39 +00:00)


In [ ]:
amia_result_df_1_deepseek.to_csv('./data/amia_result_df_1_deepseek.csv')

time: 8.72 ms (started: 2024-07-26 06:03:39 +00:00)


#### evaluation

In [ ]:
import chardet

with open('./data/amia_vali_df_new.csv', 'rb') as f:
    result = chardet.detect(f.read())
    print(result['encoding'])

Windows-1252
time: 407 ms (started: 2024-08-01 12:41:27 +00:00)


In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

90 90
90 90
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']
time: 1.7 s (started: 2024-09-01 23:28:45 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [04:58<00:00,  3.32s/it]

BERTScore for NER (Exposure): (0.80166068871816, 0.7959731251001358, 0.7956820338964462)
BERTScore for NER (Outcome Phenotype): (0.6843553612629573, 0.6815041598346498, 0.6795032017760807)
time: 10min (started: 2024-07-26 06:13:54 +00:00)


##### relationships

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 90/90 [11:45<00:00,  7.84s/it]

Accuracy of direction: 0.7815126050420168
Precision of direction: 0.72991452991453
Recall of direction: 0.7116826528591235
F1-score of direction: 0.7134606781438974
Accuracy of significance: 0.9327731092436975
Precision of significance: 0.8025062656641603
Recall of significance: 0.9427609427609428
F1-score of significance: 0.8597585690369196
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                  | outcome_pred                                  | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:----------------------------------------------|:

##### error analysis

In [ ]:
error_df = pd.merge(amia_vali_df[['pmid','full_abs']].astype(str),falsely_predicted_direction.astype(str),on='pmid',how='right')
error_df.to_excel('./data/result_df_2_deepseek_error.xlsx')
error_df

In [ ]:
falsely_predicted_direction

,pmid,exposure_gold,exposure_pred,outcome_gold,outcome_pred,direction_gold,direction_pred,significance_gold,significance_pred
5,7997845,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,diastolic blood pressure,not found,decrease,negative,negative
6,7997845,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,systolic blood pressure,not found,decrease,negative,positive
12,11252212,group office visits,group office visits,low-density lipoprotein,low-density lipoprotein reductions,increase,decrease,positive,positive
25,18055274,telephone-delivered physical activity and diet...,telephone-delivered intervention for physical ...,physical activity and dietary behavior change,physical activity and dietary behavior change,increase,not found,positive,not found
33,20823897,folinic acid supplementation,folinic acid supplementation,plasma folate concentrations,plasma folate concentrations,not found,increase,positive,positive
44,24462043,white fish consumption,regular consumption of white fish,serum LDL concentrations,serum HDL concentrations,decrease,not found,positive,negative
45,24462043,white fish consumption,regular consumption of white fish,serum LDL concentrations,serum triglyceride concentrations,decrease,not found,positive,negative
46,24462043,white fish consumption,regular consumption of white fish,serum HDL concentrations,serum LDL concentrations,not found,decrease,negative,positive
66,25421976,dietary nitrate,dietary nitrate supplementation,Endothelial function,endothelial function,decrease,increase,positive,positive
68,27223304,Paleolithic diet,Paleolithic diet,weight loss,weight loss,decrease,increase,positive,positive


time: 27.2 ms (started: 2024-09-01 23:40:37 +00:00)


In [ ]:
falsely_predicted_significance

,pmid,exposure_gold,exposure_pred,outcome_gold,outcome_pred,direction_gold,direction_pred,significance_gold,significance_pred
6,7997845,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,systolic blood pressure,not found,decrease,negative,positive
7,7997845,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,diastolic blood pressure,decrease,decrease,positive,negative
11,11252212,group office visits,group office visits,The total PMPM expenses,total PMPM expenses,decrease,decrease,positive,negative
25,18055274,telephone-delivered physical activity and diet...,telephone-delivered intervention for physical ...,physical activity and dietary behavior change,physical activity and dietary behavior change,increase,not found,positive,not found
44,24462043,white fish consumption,regular consumption of white fish,serum LDL concentrations,serum HDL concentrations,decrease,not found,positive,negative
45,24462043,white fish consumption,regular consumption of white fish,serum LDL concentrations,serum triglyceride concentrations,decrease,not found,positive,negative
46,24462043,white fish consumption,regular consumption of white fish,serum HDL concentrations,serum LDL concentrations,not found,decrease,negative,positive
86,30572894,krill powder,krill powder,adverse events,adverse events,decrease,not found,positive,negative


time: 20.9 ms (started: 2024-09-01 23:40:37 +00:00)


#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 90
90 90
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
time: 20.2 ms (started: 2024-07-30 03:14:14 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [15:30<00:00, 10.34s/it]

BERTScore for NER (Exposure): (0.7946276092792258, 0.7924612881067921, 0.7898589813095682)
BERTScore for NER (Outcome): (0.8010279609963246, 0.8070601950475268, 0.7994669290397698)
time: 22min 52s (started: 2024-07-29 04:58:36 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

  0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 90/90 [11:25<00:00,  7.62s/it]

BERTScore for NER (Exposure): (0.82238779499613, 0.8345131640290392, 0.8241106978264349)
BERTScore for NER (Outcome): (0.8174943896658692, 0.8134740135826908, 0.8098718704103114)
time: 16min 23s (started: 2024-07-30 03:15:11 +00:00)


### gpt 4 mini

In [ ]:
MODEL="gpt-4o-mini"
#openai
client = OpenAI(api_key='YOUR_API_KEY')

time: 55.8 ms (started: 2024-07-26 06:03:46 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "oral magnesium supplementation",
    "outcome": "small arterial compliance",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "endothelial function",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "exercise capacity",
    "significance": "not found",
    "direction": "not found",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "quality-of-life score",
    "significance": "not found",
    "direction": "not found",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
}
]
time: 9.74 s (started: 2024-07-26 06:03:47 +00:00)


In [ ]:
amia_result_df_1_mini = compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 90/90 [01:14<00:00,  1.21it/s]

time: 1min 14s (started: 2024-07-26 06:03:56 +00:00)


In [ ]:
print(amia_result_df_1_mini.shape)
print(amia_result_df_1_mini['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_1_mini['pmid'])]
left_df

(353, 7)
82


,pmid,article_title,full_abs,input
38,8188494,Modifying dietary and tobacco use patterns in ...,"This article describes the conceptual basis, d...",Title: Modifying dietary and tobacco use patte...
59,24500553,Innovation to reduce cardiovascular complicati...,This article describes selected primary outcom...,Title: Innovation to reduce cardiovascular com...
67,21718558,Primary prevention of CVD: diet.,INTRODUCTION: Diet is important in the cause o...,Title: Primary prevention of CVD: diet.. Abstr...
75,27646568,"Fish Consumption, Omega-3 Fatty Acids, and Ris...",INTRODUCTION: Data on omega-3 polyunsaturated ...,"Title: Fish Consumption, Omega-3 Fatty Acids, ..."
76,18783618,Antihypertensive drug class and impaired fasti...,BACKGROUND: There is a scarcity of studies add...,Title: Antihypertensive drug class and impaire...
77,34039017,Poor Masticatory Capacity and Blood Biomarkers...,Figure: see text].,Title: Poor Masticatory Capacity and Blood Bio...
86,18055274,The Logan Healthy Living Program: a cluster ra...,BACKGROUND: Physical activity and dietary beha...,Title: The Logan Healthy Living Program: a clu...
93,7832171,Perspectives on the School Nutrition Dietary A...,The School Nutrition Dietary Assessment Study ...,Title: Perspectives on the School Nutrition Di...


time: 20.4 ms (started: 2024-07-26 06:05:11 +00:00)


In [ ]:
amia_result_df_1_mini.to_csv('./data/amia_result_df_1_mini.csv')

time: 9.58 ms (started: 2024-07-26 06:05:11 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_mini.csv')

print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']
time: 1.41 s (started: 2024-08-01 12:56:37 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 82/82 [04:22<00:00,  3.21s/it]

BERTScore for NER (Exposure): (0.8467398918256527, 0.8291600913536258, 0.8354122867671455)
BERTScore for NER (Outcome Phenotype): (0.6737361757493601, 0.6645073058401666, 0.6657894918831383)
time: 8min 44s (started: 2024-07-26 06:43:56 +00:00)


##### relationships

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 82/82 [08:51<00:00,  6.48s/it]

Accuracy of direction: 0.8322147651006712
Precision of direction: 0.7799589700056594
Recall of direction: 0.8109599254426841
F1-score of direction: 0.7922583404619332
Accuracy of significance: 0.9261744966442953
Precision of significance: 0.8344723892837616
Recall of significance: 0.9379026485325699
F1-score of significance: 0.8737774355025029
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                    | outcome_pred                             | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:------------------------------------------------|

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_mini.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 82
82 82
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
time: 19.6 ms (started: 2024-07-30 04:32:10 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 82/82 [12:18<00:00,  9.01s/it]

BERTScore for NER (Exposure): (0.8256252221763134, 0.804692570996635, 0.8116116324330077)
BERTScore for NER (Outcome): (0.7970722883841709, 0.7890785183561476, 0.7893181166758663)
time: 17min 49s (started: 2024-07-30 04:33:51 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 82/82 [10:42<00:00,  7.84s/it]

BERTScore for NER (Exposure): (0.8535037205041012, 0.8798543387484328, 0.8611142557358074)
BERTScore for NER (Outcome): (0.8134346852887352, 0.8215535564242669, 0.8134625810497212)
time: 15min (started: 2024-07-30 04:51:40 +00:00)


### chatglm

In [ ]:
MODEL="glm-4-airx"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://open.bigmodel.cn/api/paas/v4/")

time: 51.4 ms (started: 2024-07-26 06:06:16 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "chronic oral magnesium supplementation",
    "outcome": "endothelial function",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
},
{
    "exposure": "chronic oral magnesium supplementation",
    "outcome": "small arterial compliance",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "heart failure",
    "comparator": "placebo"
}]
time: 3.25 s (started: 2024-07-26 06:06:18 +00:00)


In [ ]:
amia_result_df_1_glm = compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 90/90 [00:40<00:00,  2.22it/s]

time: 40.5 s (started: 2024-07-26 06:06:21 +00:00)


In [ ]:
print(amia_result_df_1_glm.shape)
print(amia_result_df_1_glm['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_1_glm['pmid'])]
left_df

(382, 7)
90


,pmid,article_title,full_abs,input


time: 13.8 ms (started: 2024-07-26 06:07:02 +00:00)


In [ ]:
amia_result_df_1_glm.to_csv('./data/amia_result_df_1_glm.csv')

time: 7.44 ms (started: 2024-07-26 06:07:02 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_glm.csv')

print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found' 'inverse']
time: 675 ms (started: 2024-08-01 13:05:30 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [04:41<00:00,  3.13s/it]

BERTScore for NER (Exposure): (0.811409854888916, 0.8295719153351254, 0.8174286699957318)
BERTScore for NER (Outcome Phenotype): (0.6774529503451453, 0.6843443473180135, 0.677845040625996)
time: 9min 33s (started: 2024-07-26 07:01:45 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 90/90 [10:05<00:00,  6.73s/it]

Accuracy of direction: 0.7766990291262136
Precision of direction: 0.7321210395834075
Recall of direction: 0.7278195488721805
F1-score of direction: 0.7264457439896037
Accuracy of significance: 0.9029126213592233
Precision of significance: 0.8583221775992861
Recall of significance: 0.81078691423519
F1-score of significance: 0.8169512966476914
Falsely predicted associations (direction):
|    |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                             | outcome_pred                                             | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|---:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:----------------------------

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_glm.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())

print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 90
90 90
['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
time: 371 ms (started: 2024-07-30 05:12:26 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [12:44<00:00,  8.49s/it]

BERTScore for NER (Exposure): (0.7847143130272812, 0.7919634467696551, 0.7840437233818244)
BERTScore for NER (Outcome): (0.765990265115859, 0.7715032368661865, 0.7644756509907662)
time: 19min 21s (started: 2024-07-30 05:12:31 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [11:04<00:00,  7.38s/it]

BERTScore for NER (Exposure): (0.8560897404777592, 0.8447179141743429, 0.8451964354206776)
BERTScore for NER (Outcome): (0.8048480532981537, 0.7933293311368852, 0.7928814579060662)
time: 15min 45s (started: 2024-07-30 05:31:52 +00:00)


### qwen

In [ ]:
MODEL="qwen-plus"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

time: 50.5 ms (started: 2024-07-26 06:07:02 +00:00)


In [ ]:
completion = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": system_1}, # <-- This is the system message that provides context to the model
    {"role": "user", "content": test_prompt},
    {"role": "user", "content": test_input_2}
  ]
)
test_result = completion.choices[0].message.content
print(test_result)

[{
    "exposure": "oral magnesium supplementation",
    "outcome": "small arterial compliance",
    "significance": "positive",
    "direction": "increase",
    "population_main_condition": "symptomatic heart failure",
    "comparator": "placebo"

},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "large arterial elasticity/compliance",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "symptomatic heart failure",
    "comparator": "placebo"

},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "hemodynamic parameters",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "symptomatic heart failure",
    "comparator": "placebo"

},
{
    "exposure": "oral magnesium supplementation",
    "outcome": "exercise capacity",
    "significance": "negative",
    "direction": "not found",
    "population_main_condition": "symptomatic heart failure",
    "comparator": "place

In [ ]:
amia_result_df_1_qwen = compile_extracted_results(input_df, 'input', prompt)

Processing abstracts: 100%|██████████| 90/90 [02:34<00:00,  1.72s/it]

time: 2min 34s (started: 2024-07-26 06:07:15 +00:00)


In [ ]:
print(amia_result_df_1_qwen.shape)
print(amia_result_df_1_qwen['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_1_qwen['pmid'])]
left_df

(444, 8)
90


,pmid,article_title,full_abs,input


time: 15.7 ms (started: 2024-07-26 06:09:50 +00:00)


In [ ]:
amia_result_df_1_qwen.to_csv('./data/amia_result_df_1_qwen.csv')

time: 8.85 ms (started: 2024-07-26 06:09:50 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_qwen.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())

print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

#pred_df['significance'] = pred_df['significance'].astype(str)
print(gold_df['significance'].value_counts(),pred_df['significance'].value_counts(),pred_df['significance'].isna().sum())

pred_df[pred_df['significance'].isna()]

90 90
90 90
['decrease' 'increase' 'not found'] ['decrease' 'not found' 'increase']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found' nan]
significance
positive     250
negative      55
not found      4
Name: count, dtype: int64 significance
positive     287
negative     139
not found     17
Name: count, dtype: int64 1


,Unnamed: 0,exposure,outcome,significance,direction,population_main_condition,comparator,pmid,signification
216,216,prescriptive exercise,cardiovascular complications (CVCs) of diabetes,NaN,not found,diabetes,not found,24500553,not found


time: 53.1 ms (started: 2025-01-15 10:47:19 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [04:28<00:00,  2.99s/it]

BERTScore for NER (Exposure): (0.8301164918475681, 0.8177357806099785, 0.8214819298850166)
BERTScore for NER (Outcome Phenotype): (0.6568801538811789, 0.6522904547552267, 0.650984149095085)
time: 8min 59s (started: 2024-07-26 07:21:07 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')
merged_df['significance_gold'] = merged_df['significance_gold'].astype(str).fillna('')
merged_df['significance_pred'] = merged_df['significance_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 90/90 [10:18<00:00,  6.87s/it]

Accuracy of direction: 0.7414965986394558
Precision of direction: 0.6914814814814815
Recall of direction: 0.7199346405228758
F1-score of direction: 0.7022133095662507
Accuracy of significance: 0.8435374149659864
Precision of significance: 0.3808580858085808
Recall of significance: 0.44661157024793385
F1-score of significance: 0.39665379665379674
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                    | outcome_pred                                    | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:----------------------------------------


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_1_qwen.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 90
90 90
['decrease' 'increase' 'not found'] ['decrease' 'not found' 'increase']
time: 343 ms (started: 2024-07-30 05:47:38 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [14:02<00:00,  9.36s/it]

BERTScore for NER (Exposure): (0.7740557363290272, 0.7585084202760708, 0.7624089839215764)
BERTScore for NER (Outcome): (0.7710959137516681, 0.771138920097126, 0.7667544819897723)
time: 20min 50s (started: 2024-07-30 05:47:39 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [11:04<00:00,  7.38s/it]

BERTScore for NER (Exposure): (0.851203942607189, 0.8660515374664602, 0.8546055401193684)
BERTScore for NER (Outcome): (0.8227692537364506, 0.8273094119606438, 0.8207947841330325)
time: 15min 45s (started: 2024-07-30 06:08:29 +00:00)


## prompt 2

In [ ]:
# Define the prompt for step 1
prompt_step1 = """
From the following given title and abstract, extract all studied exposures and cardiovascular-related outcomes separately. Ignore any other information.

Extract the following information:

1. "exposures": a list of all studied exposures
2. "outcomes": a list of all studied cardiovascular-related outcomes. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.

Return the extracted information in a JSON dictionary.

Example input:
""
Title: "Short- and long-term black tea consumption reverses endothelial dysfunction in patients with coronary artery disease."

Abstract: "BACKGROUND: Epidemiological studies suggest that tea consumption decreases cardiovascular risk, but the mechanisms of benefit remain undefined. Endothelial dysfunction has been associated with coronary artery disease and increased oxidative stress. Some antioxidants have been shown to reverse endothelial dysfunction, and tea contains antioxidant flavonoids. Methods and Results-- To test the hypothesis that tea consumption will reverse endothelial dysfunction, we randomized 66 patients with proven coronary artery disease to consume black tea and water in a crossover design. Short-term effects were examined 2 hours after consumption of 450 mL tea or water. Long-term effects were examined after consumption of 900 mL tea or water daily for 4 weeks. Vasomotor function of the brachial artery was examined at baseline and after each intervention with vascular ultrasound. Fifty patients completed the protocol and had technically suitable ultrasound measurements. Both short- and long-term tea consumption improved endothelium- dependent flow-mediated dilation of the brachial artery, whereas consumption of water had no effect (P<0.001 by repeated-measures ANOVA). Tea consumption had no effect on endothelium-independent nitroglycerin-induced dilation. An equivalent oral dose of caffeine (200 mg) had no short-term effect on flow-mediated dilation. Plasma flavonoids increased after short- and long-term tea consumption. CONCLUSIONS: Short- and long-term black tea consumption reverses endothelial vasomotor dysfunction in patients with coronary artery disease. This finding may partly explain the association between tea intake and decreased cardiovascular disease events."
""

Example output:
{
    "exposures": ["black tea consumption"],
    "outcomes": ["endothelial vasomotor dysfunction", "endothelium-dependent flow-mediated dilation of the brachial artery", "endothelium-independent nitroglycerin-induced dilation", "Plasma flavonoids"]
}

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your response.
no ```json ``` in your response.
no ```json ``` in your response.
"""

prompt_step2_template = """
From the following given title and abstract, extract the primary relationships between the given exposures and outcomes.

Use the provided entities as references.

Extract the following information:

1. "exposure": the studied exposure
2. "outcome": the studied outcome
3. "significance": only return 'positive' if there exists statistical significance or 'negative' if no significance
4. "direction": return the causal direction of the relationship (strict 3 categories): increase, decrease, or not found. (not found include relationships such as indifferent or unchanged)
5. "population_main_condition": disease of studied population
6. "comparator": compared group in relationship

Return "not found" if not found or not applicable.

Return the extracted information in a list of flat JSON dictionaries of each result.

Example input entities:
{{
    "exposures": ["black tea consumption"],
    "outcomes": ["endothelial vasomotor dysfunction", "endothelium-dependent flow-mediated dilation of the brachial artery", "endothelium-independent nitroglycerin-induced dilation", "Plasma flavonoids"]
}}

Example input title and abstract:
""
Title: "Short- and long-term black tea consumption reverses endothelial dysfunction in patients with coronary artery disease."

Abstract: "BACKGROUND: Epidemiological studies suggest that tea consumption decreases cardiovascular risk, but the mechanisms of benefit remain undefined. Endothelial dysfunction has been associated with coronary artery disease and increased oxidative stress. Some antioxidants have been shown to reverse endothelial dysfunction, and tea contains antioxidant flavonoids. Methods and Results-- To test the hypothesis that tea consumption will reverse endothelial dysfunction, we randomized 66 patients with proven coronary artery disease to consume black tea and water in a crossover design. Short-term effects were examined 2 hours after consumption of 450 mL tea or water. Long-term effects were examined after consumption of 900 mL tea or water daily for 4 weeks. Vasomotor function of the brachial artery was examined at baseline and after each intervention with vascular ultrasound. Fifty patients completed the protocol and had technically suitable ultrasound measurements. Both short- and long-term tea consumption improved endothelium- dependent flow-mediated dilation of the brachial artery, whereas consumption of water had no effect (P<0.001 by repeated-measures ANOVA). Tea consumption had no effect on endothelium-independent nitroglycerin-induced dilation. An equivalent oral dose of caffeine (200 mg) had no short-term effect on flow-mediated dilation. Plasma flavonoids increased after short- and long-term tea consumption. CONCLUSIONS: Short- and long-term black tea consumption reverses endothelial vasomotor dysfunction in patients with coronary artery disease. This finding may partly explain the association between tea intake and decreased cardiovascular disease events."
""

Example output:
[
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelial vasomotor dysfunction",
        "significance": "positive",
        "direction": "decrease",
        "population_main_condition": "coronary artery disease",
        "comparator": "water"
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelium-dependent flow-mediated dilation of the brachial artery",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "coronary artery disease",
        "comparator": "water"
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelium-independent nitroglycerin-induced dilation",
        "significance": "negative",
        "direction": "not found",
        "population_main_condition": "coronary artery disease",
        "comparator": "water"
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "Plasma flavonoids",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "coronary artery disease",
        "comparator": "water"
    }}
]

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your response.
no ```json ``` in your response.
no ```json ``` in your response.
"""

time: 451 µs (started: 2024-09-03 03:18:36 +00:00)


In [ ]:
# for AMIA salt
prompt_step1 = """
From the following given title and abstract, extract all studied exposures and cardiovascular-related outcomes separately. Ignore any other information.

Extract the following information:

1. "exposures": a list of all studied exposures
2. "outcomes": a list of all studied cardiovascular-related outcomes. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.

Return the extracted information in a JSON dictionary.

Example input:
""
Title: "Short- and long-term black tea consumption reverses endothelial dysfunction in patients with coronary artery disease."

Abstract: "BACKGROUND: Epidemiological studies suggest that tea consumption decreases cardiovascular risk, but the mechanisms of benefit remain undefined. Endothelial dysfunction has been associated with coronary artery disease and increased oxidative stress. Some antioxidants have been shown to reverse endothelial dysfunction, and tea contains antioxidant flavonoids. Methods and Results-- To test the hypothesis that tea consumption will reverse endothelial dysfunction, we randomized 66 patients with proven coronary artery disease to consume black tea and water in a crossover design. Short-term effects were examined 2 hours after consumption of 450 mL tea or water. Long-term effects were examined after consumption of 900 mL tea or water daily for 4 weeks. Vasomotor function of the brachial artery was examined at baseline and after each intervention with vascular ultrasound. Fifty patients completed the protocol and had technically suitable ultrasound measurements. Both short- and long-term tea consumption improved endothelium- dependent flow-mediated dilation of the brachial artery, whereas consumption of water had no effect (P<0.001 by repeated-measures ANOVA). Tea consumption had no effect on endothelium-independent nitroglycerin-induced dilation. An equivalent oral dose of caffeine (200 mg) had no short-term effect on flow-mediated dilation. Plasma flavonoids increased after short- and long-term tea consumption. CONCLUSIONS: Short- and long-term black tea consumption reverses endothelial vasomotor dysfunction in patients with coronary artery disease. This finding may partly explain the association between tea intake and decreased cardiovascular disease events."
""

Example output:
{
    "exposures": ["black tea consumption"],
    "outcomes": ["endothelial vasomotor dysfunction", "endothelium-dependent flow-mediated dilation of the brachial artery", "endothelium-independent nitroglycerin-induced dilation", "Plasma flavonoids"]
}

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your response.
no ```json ``` in your response.
no ```json ``` in your response.
"""

prompt_step2_template = """
From the following given title and abstract, extract the primary relationships between the given exposures and outcomes.

Use the provided entities as references.

Extract the following information:

1. "exposure": the studied exposure
2. "outcome": the studied outcome
3. "significance": only return 'positive' if there exists statistical significance or 'negative' if no significance
4. "direction": return the causal direction of the relationship (strict 3 categories): increase, decrease, or not found. (not found include relationships such as indifferent or unchanged)
5. "population_main_condition": disease of studied population
6. "comparator": compared group in relationship
7. study_design (only 1 of 5 strict abbreiviation categories: Meta analysis(META), Randomized controlled trial(RCT), (Mendelian randomization)MR, Observational study(OS), Not identified);
8. included studies (only return number + RCT/MR/OS)(only for META, strict null for others(RCT,MR,OS));
9. number of participants (numeric number only)

Return "not found" if not found or not applicable.

Return the extracted information in a list of flat JSON dictionaries of each result.

Example input entities:
{{
    "exposures": ["black tea consumption"],
    "outcomes": ["endothelial vasomotor dysfunction", "endothelium-dependent flow-mediated dilation of the brachial artery", "endothelium-independent nitroglycerin-induced dilation", "Plasma flavonoids"]
}}

Example input title and abstract:
""
Title: "Short- and long-term black tea consumption reverses endothelial dysfunction in patients with coronary artery disease."

Abstract: "BACKGROUND: Epidemiological studies suggest that tea consumption decreases cardiovascular risk, but the mechanisms of benefit remain undefined. Endothelial dysfunction has been associated with coronary artery disease and increased oxidative stress. Some antioxidants have been shown to reverse endothelial dysfunction, and tea contains antioxidant flavonoids. Methods and Results-- To test the hypothesis that tea consumption will reverse endothelial dysfunction, we randomized 66 patients with proven coronary artery disease to consume black tea and water in a crossover design. Short-term effects were examined 2 hours after consumption of 450 mL tea or water. Long-term effects were examined after consumption of 900 mL tea or water daily for 4 weeks. Vasomotor function of the brachial artery was examined at baseline and after each intervention with vascular ultrasound. Fifty patients completed the protocol and had technically suitable ultrasound measurements. Both short- and long-term tea consumption improved endothelium- dependent flow-mediated dilation of the brachial artery, whereas consumption of water had no effect (P<0.001 by repeated-measures ANOVA). Tea consumption had no effect on endothelium-independent nitroglycerin-induced dilation. An equivalent oral dose of caffeine (200 mg) had no short-term effect on flow-mediated dilation. Plasma flavonoids increased after short- and long-term tea consumption. CONCLUSIONS: Short- and long-term black tea consumption reverses endothelial vasomotor dysfunction in patients with coronary artery disease. This finding may partly explain the association between tea intake and decreased cardiovascular disease events."
""

Example output:
[
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelial vasomotor dysfunction",
        "significance": "positive",
        "direction": "decrease",
        "population_main_condition": "coronary artery disease",
        "comparator": "water",
        "study_design": "RCT",
        "included_studies": "null",
        "number_of_participants": 66
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelium-dependent flow-mediated dilation of the brachial artery",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "coronary artery disease",
        "comparator": "water",
        "study_design": "RCT",
        "included_studies": "null",
        "number_of_participants": 66
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "endothelium-independent nitroglycerin-induced dilation",
        "significance": "negative",
        "direction": "not found",
        "population_main_condition": "coronary artery disease",
        "comparator": "water",
        "study_design": "RCT",
        "included_studies": "null",
        "number_of_participants": 66
    }},
    {{
        "exposure": "black tea consumption",
        "outcome": "Plasma flavonoids",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "coronary artery disease",
        "comparator": "water",
        "study_design": "RCT",
        "included_studies": "null",
        "number_of_participants": 66
    }}
]

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your response.
no ```json ``` in your response.
no ```json ``` in your response.
"""

time: 1.15 ms (started: 2024-09-03 03:26:15 +00:00)


In [ ]:
import openai
import pandas as pd
from tqdm import tqdm
from ast import literal_eval
import threading
from queue import Queue
import json

# Function to perform the two-step extraction
def two_step_extraction(text):
    # Step 1: Extract exposures and outcomes
    completion_step1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in social determinatants of health (SDOH) and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
            {"role": "user", "content": prompt_step1},
            {"role": "user", "content": text}
        ]
    )
    entities_result = completion_step1.choices[0].message.content
    entities_result = entities_result.strip().removeprefix("```json").removesuffix("```").strip()
    entities_dict = json.loads(entities_result)
    entities_json = json.dumps(entities_dict, indent=2)

    # Prepare step 2 prompt
    prompt_step2 = prompt_step2_template.format(entities=entities_json)

    # Step 2: Extract associations using the entities
    completion_step2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in social determinatants of health (SDOH) and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
            {"role": "user", "content": prompt_step2},
            {"role": "user", "content": f"extracted entities: {entities_json}"},
            {"role": "user", "content": f"title and abstract: {text}"}
        ]
    )
    associations_result = completion_step2.choices[0].message.content

    return associations_result

# Worker function to process each row
def worker(input_queue, output_queue, text_column, progress_bar):
    while True:
        index, row = input_queue.get()
        if index is None:
            break
        try:
            output = two_step_extraction(row[text_column])
            extracted_results = literal_eval(output)
            for res in extracted_results:
                res['pmid'] = row['pmid']
                output_queue.put(res)
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            print(f"Input text: {row[text_column][:500]}...")  # Print the first 500 characters of the text
            print(f"Output: {output}")  # Print the output if available
        finally:
            input_queue.task_done()
            progress_bar.update(1)

# Main function to compile extracted results
def compile_extracted_results(input_df, text_column):
    results = []
    input_queue = Queue()
    output_queue = Queue()

    # Progress bar for processing
    progress_bar = tqdm(total=input_df.shape[0], desc="Processing abstracts")

    # Start worker threads
    for _ in range(8):
        threading.Thread(target=worker, args=(input_queue, output_queue, text_column, progress_bar), daemon=True).start()

    # Enqueue rows for processing
    for index, row in input_df.iterrows():
        input_queue.put((index, row))

    # Wait for all tasks to be processed
    input_queue.join()

    # Collect results from output queue
    while not output_queue.empty():
        results.append(output_queue.get())

    # Close progress bar
    progress_bar.close()

    results_df = pd.json_normalize(results)
    return results_df

time: 2.58 ms (started: 2024-09-03 03:18:37 +00:00)


### deepseek

In [ ]:
MODEL="deepseek-chat"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://api.deepseek.com")

time: 53.6 ms (started: 2024-09-03 03:18:39 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH) and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH)  and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_2}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["dietary advice and self monitoring of chloride concentration in urine", "moderately salt restricted diet"],
    "outcomes": ["blood pressure", "diastolic blood pressure", "systolic blood pressure", "24 h sodium excretion"]
}
Associations extracted: [
    {
        "exposure": "dietary advice and self monitoring of chloride concentration in urine",
        "outcome": "blood pressure",
        "significance": "not found",
        "direction": "not found",
        "population_main_condition": "heart failure",
        "comparator": "not found"
    },
    {
        "exposure": "dietary advice and self monitoring of chloride concentration in urine",
        "outcome": "diastolic blood pressure",
        "significance": "not found",
        "direction": "not found",
        "population_main_condition": "heart failure",
        "comparator": "not found"
    },
    {
        "exposure": "dietary advice and self monitoring of chloride concentration in uri

In [ ]:
amia_result_df_2_deepseek = compile_extracted_results(input_df, 'input')


Processing abstracts:  98%|█████████▊| 88/90 [05:44<00:40, 20.08s/it]

Error processing row 52: '{' was never closed (<unknown>, line 434)
Input text: Title: Changes in weight loss, body composition and cardiovascular disease risk after altering macronutrient distributions during a regular exercise program in obese women.. Abstract: BACKGROUND: This study's purpose investigated the impact of different macronutrient distributions and varying caloric intakes along with regular exercise for metabolic and physiological changes related to weight loss. METHODS: One hundred forty-one sedentary, obese women (38.7 ± 8.0 yrs, 163.3 ± 6.9 cm, 93.2 ± 16.5...
Output: [
    {
        "exposure": "high-energy diet",
        "outcome": "weight loss",
        "significance": "positive",
        "direction": "decrease",
        "population_main_condition": "obesity",
        "comparator": "no diet + no exercise control group"
    },
    {
        "exposure": "high-energy diet",
        "outcome": "body composition",
        "significance": "positive",
        "direction": 


Processing abstracts: 100%|██████████| 90/90 [05:52<00:00,  3.91s/it]

time: 5min 52s (started: 2024-07-26 08:09:02 +00:00)


In [ ]:
print(amia_result_df_2_deepseek.shape)
print(amia_result_df_2_deepseek['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_2_deepseek['pmid'])]
left_df

(419, 7)
89


,pmid,article_title,full_abs,input
52,21092228,"Changes in weight loss, body composition and c...",BACKGROUND: This study's purpose investigated ...,"Title: Changes in weight loss, body compositio..."


time: 23 ms (started: 2024-07-26 08:14:54 +00:00)


In [ ]:
amia_result_df_2_deepseek.to_csv('./data/amia_result_df_2_deepseek.csv')

time: 8.82 ms (started: 2024-07-26 08:14:54 +00:00)


#### evaluation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

90 89
89 89
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']
time: 1.83 s (started: 2024-09-02 05:09:14 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)


Processing abstracts:   9%|▉         | 8/90 [13:45<2:21:02, 103.20s/it]

100%|██████████| 89/89 [03:49<00:00,  2.58s/it]

BERTScore for NER (Exposure): (0.8266565849941768, 0.8108968610843915, 0.8156460336085116)
BERTScore for NER (Outcome Phenotype): (0.65879077137856, 0.6411778200208471, 0.6466201443350716)
time: 7min 40s (started: 2024-07-26 08:15:04 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

  0%|          | 0/89 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

100%|██████████| 89/89 [12:07<00:00,  8.18s/it]

Accuracy of direction: 0.8434782608695652
Precision of direction: 0.8228713070275341
Recall of direction: 0.824385996517144
F1-score of direction: 0.8207984833277573
Accuracy of significance: 0.9565217391304348
Precision of significance: 0.9382716049382717
Recall of significance: 0.9818840579710145
F1-score of significance: 0.9566754075932048
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                    | outcome_pred                             | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:------------------------------------------------|:

##### error analysis

In [ ]:
error_df_direction = pd.merge(amia_vali_df[['pmid','full_abs']].astype(str),falsely_predicted_direction.astype(str),on='pmid',how='right')
error_df_direction.to_excel('./data/amia_result_df_2_deepseek_direction.xlsx')
error_df_direction

,pmid,full_abs,exposure_gold,exposure_pred,outcome_gold,outcome_pred,direction_gold,direction_pred,significance_gold,significance_pred
0,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,diastolic blood pressure,not found,decrease,negative,negative
1,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,diastolic blood pressure,not found,decrease,negative,negative
2,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,diastolic blood pressure,not found,decrease,negative,negative
3,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,systolic blood pressure,not found,decrease,negative,negative
4,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,systolic blood pressure,not found,decrease,negative,negative
5,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,diastolic blood pressure,systolic blood pressure,not found,decrease,negative,negative
6,11252212,BACKGROUND: We evaluated the effectiveness of ...,group office visits,group office visits,low-density lipoprotein,low-density lipoprotein reductions,increase,decrease,positive,positive
7,11252212,BACKGROUND: We evaluated the effectiveness of ...,group office visits,group office visits,low-density lipoprotein,low-density lipoprotein reductions,increase,decrease,positive,positive
8,11252212,BACKGROUND: We evaluated the effectiveness of ...,group office visits,group office visits,low-density lipoprotein,low-density lipoprotein reductions,increase,decrease,positive,positive
9,11252212,BACKGROUND: We evaluated the effectiveness of ...,group office visits,group office visits,low-density lipoprotein,low-density lipoprotein reductions,increase,decrease,positive,positive


time: 847 ms (started: 2024-09-02 05:21:34 +00:00)


In [ ]:
error_df_sig = pd.merge(amia_vali_df[['pmid','full_abs']].astype(str),falsely_predicted_significance.astype(str),on='pmid',how='right')
error_df_sig.to_excel('./data/amia_result_df_2_deepseek_sig.xlsx')
error_df_sig

,pmid,full_abs,exposure_gold,exposure_pred,outcome_gold,outcome_pred,direction_gold,direction_pred,significance_gold,significance_pred
0,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,diastolic blood pressure,decrease,decrease,positive,negative
1,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,diastolic blood pressure,decrease,decrease,positive,negative
2,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,diastolic blood pressure,decrease,decrease,positive,negative
3,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,systolic blood pressure,decrease,decrease,positive,negative
4,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,systolic blood pressure,decrease,decrease,positive,negative
5,7997845,The aims of this present study were firstly to...,dietary advice combined with self monitoring o...,dietary advice combined with self monitoring o...,systolic blood pressure,systolic blood pressure,decrease,decrease,positive,negative
6,18160764,It has been reported that green tea consumptio...,Tea catechin consumption,catechin consumption,Plasma oxidized LDL,plasma LDL-C,decrease,not found,positive,negative
7,18160764,It has been reported that green tea consumptio...,Tea catechin consumption,catechin consumption,Plasma oxidized LDL,plasma LDL-C,decrease,not found,positive,negative
8,18160764,It has been reported that green tea consumptio...,Tea catechin consumption,catechin consumption,Plasma oxidized LDL,plasma LDL-C,decrease,not found,positive,negative
9,18160764,It has been reported that green tea consumptio...,Tea catechin consumption,catechin consumption,Plasma oxidized LDL,plasma LDL-C,decrease,not found,positive,negative


time: 675 ms (started: 2024-09-02 05:21:35 +00:00)


#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_deepseek.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 89
89 89
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
time: 15.3 ms (started: 2024-07-30 06:24:14 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [13:34<00:00,  9.15s/it]

BERTScore for NER (Exposure): (0.8133035263289576, 0.7947711700546569, 0.8009747731080954)
BERTScore for NER (Outcome): (0.7507923003452928, 0.7377372544481239, 0.7402996306710724)
time: 19min 9s (started: 2024-07-30 06:24:14 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [10:52<00:00,  7.33s/it]

BERTScore for NER (Exposure): (0.8421378500826724, 0.8616994818051656, 0.848360768578074)
BERTScore for NER (Outcome): (0.7812050793931555, 0.7988197268612349, 0.7858537236756798)
time: 15min 22s (started: 2024-07-30 06:43:24 +00:00)


### gpt 4mini

In [ ]:
print('aa')

aa
time: 525 µs (started: 2024-07-26 08:33:08 +00:00)


In [ ]:
MODEL="gpt-4o-mini"
#openai
client = OpenAI(api_key='YOUR_API_KEY')

time: 54.8 ms (started: 2024-07-26 08:33:25 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH) and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH)  and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_2}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["salt restriction", "dietary advice", "self monitoring of chloride concentration in urine"],
    "outcomes": ["blood pressure", "systolic blood pressure", "diastolic blood pressure", "24 h sodium excretion"]
}
Associations extracted: [
    {
        "exposure": "magnesium supplementation",
        "outcome": "endothelial function",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "symptomatic heart failure",
        "comparator": "placebo"
    },
    {
        "exposure": "magnesium supplementation",
        "outcome": "small arterial compliance",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "symptomatic heart failure",
        "comparator": "placebo"
    },
    {
        "exposure": "magnesium supplementation",
        "outcome": "exercise capacity",
        "significance": "not found",
        "direction": "not found",
        "

In [ ]:
amia_result_df_2_mini = compile_extracted_results(input_df, 'input')

Processing abstracts: 100%|██████████| 90/90 [01:46<00:00,  1.18s/it]

time: 1min 46s (started: 2024-07-26 08:33:32 +00:00)


In [ ]:
print(amia_result_df_2_mini.shape)
print(amia_result_df_2_mini['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_2_mini['pmid'])]
left_df

(513, 7)
89


,pmid,article_title,full_abs,input
93,7832171,Perspectives on the School Nutrition Dietary A...,The School Nutrition Dietary Assessment Study ...,Title: Perspectives on the School Nutrition Di...


time: 18.6 ms (started: 2024-07-26 08:35:19 +00:00)


In [ ]:
amia_result_df_2_mini.to_csv('./data/amia_result_df_2_mini.csv')

time: 8.16 ms (started: 2024-07-26 08:35:19 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_mini.csv')

print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['significance'].unique(),pred_df['significance'].unique())

['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']
time: 655 ms (started: 2024-08-01 13:35:58 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [03:49<00:00,  2.58s/it]

BERTScore for NER (Exposure): (0.8155448915583364, 0.7941804940111181, 0.8017413311460045)
BERTScore for NER (Outcome Phenotype): (0.6518315547302868, 0.6286671051148618, 0.6365167642577311)
time: 7min 36s (started: 2024-07-26 08:35:29 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 89/89 [10:06<00:00,  6.81s/it]

Accuracy of direction: 0.8672566371681416
Precision of direction: 0.8127774186597715
Recall of direction: 0.8144887845592071
F1-score of direction: 0.8130091185410334
Accuracy of significance: 0.9734513274336283
Precision of significance: 0.9548611111111112
Recall of significance: 0.9709049255441008
F1-score of significance: 0.9625605883336119
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                              | exposure_pred                     | outcome_gold                         | outcome_pred                    | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------|:----------------------------------|:-------------------------------------|:--------------------------------|:-----------------|:-----------------|:--------------------|:--------------------|
|  10 |  6238910 | salt depletion                             | salt depletion                    | 

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_mini.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 89
89 89
['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
time: 360 ms (started: 2024-07-30 06:58:47 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [14:38<00:00,  9.87s/it]

BERTScore for NER (Exposure): (0.7769580548898333, 0.7405634904872476, 0.7543654352254261)
BERTScore for NER (Outcome): (0.7456681951261294, 0.7280287401250195, 0.7322338060617777)
time: 21min 40s (started: 2024-07-30 06:58:48 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [11:03<00:00,  7.45s/it]

BERTScore for NER (Exposure): (0.830382697219434, 0.8689909061659937, 0.8454196313153143)
BERTScore for NER (Outcome): (0.7843016960712916, 0.8098119857775814, 0.7918148595456254)
time: 15min 43s (started: 2024-07-30 07:20:28 +00:00)


### chatglm

In [ ]:
MODEL="glm-4-airx"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://open.bigmodel.cn/api/paas/v4/")

time: 79.9 ms (started: 2024-07-26 08:51:36 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH) and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH)  and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_2}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["salt restriction", "dietary advice", "self monitoring of chloride concentration in urine", "moderately salt restricted diet"],
    "outcomes": ["diet compliance", "blood pressure", "diastolic blood pressure", "systolic blood pressure", "24 h sodium excretion", "morning urine chloride concentration"]
}
Associations extracted: [
    {
        "exposure": "chronic oral magnesium supplementation",
        "outcome": "endothelial function",
        "significance": "positive",
        "direction": "increase",
        "population_main_condition": "symptomatic heart failure",
        "comparator": "placebo"
    },
    {
        "exposure": "chronic oral magnesium supplementation",
        "outcome": "exercise capacity",
        "significance": "not found",
        "direction": "not found",
        "population_main_condition": "symptomatic heart failure",
        "comparator": "placebo"
    },
    {
        "exposure": "chronic oral magnesium supplementa

In [ ]:
amia_result_df_2_glm = compile_extracted_results(input_df, 'input')

Processing abstracts:  31%|███       | 28/90 [00:34<00:40,  1.54it/s]

Error processing row 23: unterminated string literal (detected at line 100) (<unknown>, line 100)
Input text: Title: Circulating inflammatory and atherogenic biomarkers are not increased following single meals of dairy foods.. Abstract: BACKGROUND/OBJECTIVES: Inflammation characterizes obesity and is nutritionally modifiable. The hypothesis of this study is that full-fat dairy foods influence circulating inflammatory and atherogenic biomarkers according to fermentation status. SUBJECTS/METHODS: Thirteen overweight subjects participated in five test meals. Single breakfasts containing control low-fat mil...
Output: [
    {
        "exposure": "single meals of dairy foods",
        "outcome": "circulating inflammatory markers interleukin (IL)-6",
        "significance": "negative",
        "direction": "not found",
        "population_main_condition": "overweight",
        "comparator": "control low-fat milk"
    },
    {
        "exposure": "single meals of dairy foods",
        "outcom

Processing abstracts:  72%|███████▏  | 65/90 [01:09<00:28,  1.12s/it]

Error processing row 67: unterminated string literal (detected at line 108) (<unknown>, line 108)
Input text: Title: Primary prevention of CVD: diet.. Abstract: INTRODUCTION: Diet is important in the cause of many chronic diseases. Individual change in dietary behaviour has the potential to decrease the burden of chronic disease, particularly cardiovascular disease (CVD). METHODS AND OUTCOMES: We conducted a systematic review and aimed to answer the following clinical question: What are the effects of dietary advice in generally healthy adults without existing CVD or increased CVD risk factors to improv...
Output: [
    {
        "exposure": "dietary advice to increase fibre intake",
        "outcome": "cardiovascular outcomes",
        "significance": "not found",
        "direction": "not found",
        "population_main_condition": "generally healthy adults without existing CVD or increased CVD risk factors",
        "comparator": "not applicable"
    },
    {
        "exposure": "d

Processing abstracts:  74%|███████▍  | 67/90 [01:10<00:19,  1.18it/s]

Error processing row 77: invalid syntax (<unknown>, line 1)
Input text: Title: Poor Masticatory Capacity and Blood Biomarkers of Elevated Cardiovascular Disease Risk in the Community: The Paris Prospective Study III.. Abstract: Figure: see text]....
Output: ```json
[
    {
        "exposure": "Poor Masticatory Capacity",
        "outcome": "Cardiovascular Disease Risk",
        "significance": "not found",
        "direction": "not found",
        "population_main_condition": "not found",
        "comparator": "not found"
    }
]
```


... (2 outputs left out)

Processing abstracts:  99%|█████████▉| 89/90 [01:34<00:01,  1.29s/it]

Error processing row 94: '[' was never closed (<unknown>, line 1)
Input text: Title: Metabolic effects of weight loss on a very-low-carbohydrate diet compared with an isocaloric high-carbohydrate diet in abdominally obese subjects.. Abstract: OBJECTIVES: This study was designed to compare the effects of an energy-reduced, isocaloric very-low-carbohydrate, high-fat (VLCHF) diet and a high-carbohydrate, low-fat (HCLF) diet on weight loss and cardiovascular disease (CVD) risk outcomes. BACKGROUND: Despite the popularity of the VLCHF diet, no studies have compared the chronic...
Output: [
    {
        "exposure": "energy-reduced, isocaloric very-low-carbohydrate, high-fat (VLCHF) diet",
        "outcome": "weight loss",
        "significance": "positive",
        "direction": "decrease",
        "population_main_condition": "abdominally obese",
        "comparator": "isocaloric high-carbohydrate, low-fat (HCLF) diet"
    },
    {
        "exposure": "energy-reduced, isocaloric very-low-ca

Processing abstracts: 100%|██████████| 90/90 [01:39<00:00,  1.10s/it]

Error processing row 99: '{' was never closed (<unknown>, line 122)
Input text: Title: Chinese lacto-vegetarian diet exerts favorable effects on metabolic parameters, intima-media thickness, and cardiovascular risks in healthy men.. Abstract: BACKGROUND: To investigate whether the Chinese lacto-vegetarian diet has protective effects on metabolic and cardiovascular disease (CVD). METHODS: One hundred sixty-nine healthy Chinese lacto-vegetarians and 126 healthy omnivore men aged 21-76 years were enrolled. Anthropometric indexes, lipid profile, insulin sensitivity, pancreatic ...
Output: [
    {
        "exposure": "Chinese lacto-vegetarian diet",
        "outcome": "metabolic parameters",
        "significance": "positive",
        "direction": "decrease",
        "population_main_condition": "healthy",
        "comparator": "omnivore men"
    },
    {
        "exposure": "Chinese lacto-vegetarian diet",
        "outcome": "intima-media thickness",
        "significance": "positive",
   

In [ ]:
print(amia_result_df_2_glm.shape)
print(amia_result_df_2_glm['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_2_glm['pmid'])]
left_df

(491, 7)
84


,pmid,article_title,full_abs,input
23,21811291,Circulating inflammatory and atherogenic bioma...,BACKGROUND/OBJECTIVES: Inflammation characteri...,Title: Circulating inflammatory and atherogeni...
67,21718558,Primary prevention of CVD: diet.,INTRODUCTION: Diet is important in the cause o...,Title: Primary prevention of CVD: diet.. Abstr...
77,34039017,Poor Masticatory Capacity and Blood Biomarkers...,Figure: see text].,Title: Poor Masticatory Capacity and Blood Bio...
84,32958070,Effects of whey protein on glycemic control an...,BACKGROUND: This systematic review and meta-an...,Title: Effects of whey protein on glycemic con...
94,18174038,Metabolic effects of weight loss on a very-low...,OBJECTIVES: This study was designed to compare...,Title: Metabolic effects of weight loss on a v...
99,22412169,Chinese lacto-vegetarian diet exerts favorable...,BACKGROUND: To investigate whether the Chinese...,Title: Chinese lacto-vegetarian diet exerts fa...


time: 22.3 ms (started: 2024-07-26 08:53:22 +00:00)


In [ ]:
amia_result_df_2_glm.to_csv('./data/amia_result_df_2_glm.csv')

time: 7.4 ms (started: 2024-07-26 08:53:22 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_glm.csv')

print(gold_df['direction'].unique(),pred_df['direction'].unique())

['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
time: 803 ms (started: 2024-08-01 13:46:05 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 84/84 [03:38<00:00,  2.61s/it]

BERTScore for NER (Exposure): (0.7593250299493471, 0.7691300305582228, 0.7610872653978211)
BERTScore for NER (Outcome Phenotype): (0.6583364410769372, 0.6632891439256214, 0.6577893318164916)
time: 7min 16s (started: 2024-07-26 08:53:32 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 84/84 [09:40<00:00,  6.92s/it]

Accuracy of direction: 0.8289473684210527
Precision of direction: 0.7982456140350878
Recall of direction: 0.8298245614035088
F1-score of direction: 0.8097265221878226
Accuracy of significance: 0.868421052631579
Precision of significance: 0.6444444444444445
Recall of significance: 0.9470899470899471
F1-score of significance: 0.7212643678160919
Falsely predicted associations (direction):
|    |     pmid | exposure_gold                                                           | exposure_pred                                                           | outcome_gold                                    | outcome_pred                                  | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|---:|---------:|:------------------------------------------------------------------------|:------------------------------------------------------------------------|:------------------------------------------------|:----------------------------------------------|:-

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_glm.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 84
84 84
['decrease' 'increase' 'not found'] ['decrease' 'increase' 'not found']
time: 344 ms (started: 2024-07-30 07:36:11 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 84/84 [14:21<00:00, 10.25s/it]

BERTScore for NER (Exposure): (0.7266168198185766, 0.7338127333247019, 0.726110850444097)
BERTScore for NER (Outcome): (0.7256860574760006, 0.7259431625163909, 0.7216825014148645)
time: 21min 6s (started: 2024-07-30 07:36:11 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 84/84 [09:33<00:00,  6.83s/it]

BERTScore for NER (Exposure): (0.8027913419999809, 0.7985577098677091, 0.7965535312612481)
BERTScore for NER (Outcome): (0.8088105728818198, 0.8054911807432013, 0.8030412814627259)
time: 13min 53s (started: 2024-07-30 07:57:18 +00:00)


### qwen

In [ ]:
MODEL="qwen-plus"
client = OpenAI(api_key='YOUR_API_KEY',base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

time: 54 ms (started: 2024-07-26 09:08:02 +00:00)


In [ ]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH) and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in social determinants of health (SDOH)  and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_2}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["dietary advice", "self monitoring of chloride concentration in urine"],
    "outcomes": ["blood pressure", "24 h sodium excretion", "morning urine chloride concentration"]
}
Associations extracted: [{"exposure": "dietary advice", "outcome": "blood pressure", "significance": "not found", "direction": "not found", "population_main_condition": "symptomatic heart failure", "comparator": "placebo"}, {"exposure": "dietary advice", "outcome": "24 h sodium excretion", "significance": "not found", "direction": "not found", "population_main_condition": "symptomatic heart failure", "comparator": "placebo"}, {"exposure": "dietary advice", "outcome": "morning urine chloride concentration", "significance": "not found", "direction": "not found", "population_main_condition": "symptomatic heart failure", "comparator": "placebo"}, {"exposure": "oral magnesium supplementation", "outcome": "blood pressure", "significance": "not found", "direction": "not found", "po

In [ ]:
amia_result_df_2_qwen = compile_extracted_results(input_df, 'input')

Processing abstracts: 100%|██████████| 90/90 [04:31<00:00,  3.02s/it]

time: 4min 31s (started: 2024-07-26 09:08:24 +00:00)


In [ ]:
print(amia_result_df_2_qwen.shape)
print(amia_result_df_2_qwen['pmid'].nunique())
left_df = input_df[~input_df['pmid'].isin(amia_result_df_2_qwen['pmid'])]
left_df

(556, 7)
90


,pmid,article_title,full_abs,input


time: 13.9 ms (started: 2024-07-26 09:12:55 +00:00)


In [ ]:
amia_result_df_2_qwen.to_csv('./data/amia_result_df_2_qwen.csv')

time: 8.69 ms (started: 2024-07-26 09:12:55 +00:00)


#### evaLuation

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df_new.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_qwen.csv')

print(gold_df['direction'].unique(),pred_df['direction'].unique())
print(gold_df['direction'].value_counts(),pred_df['direction'].value_counts())

print(gold_df['significance'].unique(),pred_df['significance'].unique())
print(gold_df['significance'].value_counts(),pred_df['significance'].value_counts())

['decrease' 'increase' 'not found'] ['increase' 'not found' 'decrease' 'improve']
direction
decrease     148
increase     101
not found     60
Name: count, dtype: int64 direction
decrease     222
not found    221
increase     109
improve        4
Name: count, dtype: int64
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']
significance
positive     250
negative      55
not found      4
Name: count, dtype: int64 significance
positive     315
negative     217
not found     24
Name: count, dtype: int64
time: 44.8 ms (started: 2025-01-15 09:45:38 +00:00)


##### entities

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [03:56<00:00,  2.63s/it]

BERTScore for NER (Exposure): (0.8355225046475728, 0.8183276842037837, 0.82422859635618)
BERTScore for NER (Outcome Phenotype): (0.64520280311505, 0.6361963593297535, 0.6373595893383026)
time: 7min 50s (started: 2024-07-26 09:13:06 +00:00)


##### r

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

  0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

100%|██████████| 90/90 [17:07<00:00, 11.42s/it]

Accuracy of direction: 0.7769784172661871
Precision of direction: 0.5529132326007327
Recall of direction: 0.5523030942334739
F1-score of direction: 0.5510454794362625
Accuracy of significance: 0.7841726618705036
Precision of significance: 0.5640554298642534
Recall of significance: 0.6591478696741855
F1-score of significance: 0.5870603975535168
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                | exposure_pred                                | outcome_gold                                                                 | outcome_pred                                                | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:---------------------------------------------|:---------------------------------------------|:-----------------------------------------------------------------------------|:------------------------------------------------------------|:-----------------|:-

#### evaluation_new

In [ ]:
gold_df = pd.read_csv('./data/amia_vali_df.csv')
pred_df = pd.read_csv('./data/amia_result_df_2_qwen.csv')

print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())
gold_df = gold_df[gold_df['pmid'].isin(pred_df['pmid'])]
print(gold_df['pmid'].nunique(),pred_df['pmid'].nunique())


print(gold_df['direction'].unique(),pred_df['direction'].unique())

90 90
90 90
['decrease' 'increase' 'not found'] ['increase' 'not found' 'decrease' 'improve']
time: 761 ms (started: 2025-01-15 08:25:36 +00:00)


###### entities

In [ ]:
# extracted to gold
gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for pred_text in pred_texts:
                P, R, F1 = score([pred_text], [gold_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [15:05<00:00, 10.06s/it]

BERTScore for NER (Exposure): (0.7827953643228379, 0.7527459786713488, 0.7633477279379324)
BERTScore for NER (Outcome): (0.7471465399508835, 0.7365200293090154, 0.7374707801437747)
time: 21min 42s (started: 2024-07-30 08:11:12 +00:00)


In [ ]:
#gold to extracted

gold_standard = gold_df[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df[['pmid', 'exposure', 'outcome']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cuda')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome):", bertscore_ner_outcome_avg)

100%|██████████| 90/90 [11:06<00:00,  7.41s/it]

BERTScore for NER (Exposure): (0.8419052840306841, 0.8725098171110811, 0.8527627453721803)
BERTScore for NER (Outcome): (0.7949020805267187, 0.8069426201202057, 0.795682318071961)
time: 15min 49s (started: 2024-07-30 08:32:54 +00:00)


# Test Number 2 July 29.